# SETUP, CONFIG & DATASET

## Section 01 : CONFIG

**Step 01 — Install dependencies.** Install the libraries used by the data pipeline, models, plots, and app.

In [ ]:
# Install dependencies
# Imports
!pip install -q tifffile timm grad-cam imagecodecs gradio opencv-python

**Step 02 — Configure the project.** Set paths, four target classes, split ratios, preprocessing, training parameters, seed, and output folders.

In [ ]:
# Configure the project
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

@dataclass
class Config:
    # ---- paths ----
    project_dir: str = "/content/drive/MyDrive/FYP"
    local_data_dir: str = "/content/nastar_local"   # fast local disk copy

    # ---- data (DataLoader attributes) ----
    class_names: list = field(default_factory=lambda: ["Cargo", "Tanker", "Fishing", "Passenger"])
    quality_filter: bool = True
    group_key: str = "MMSI"          # keeps all patches of one vessel in one split
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    n_folds: int = 5

    # ---- preprocessing (Preprocessor attributes) ----
    target_size: int = 256
    crop_mode: str = "resize"        # "resize" or "center_crop"
    intensity_mode: str = "log"      # "log" or "percentile"
    aug_enabled: bool = True

    # ---- training (BaseModel attributes) ----
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    batch_size: int = 16
    max_epochs: int = 100
    patience: int = 12
    label_smoothing: float = 0.0

    # ---- swin ----
    swin_input_size: int = 224
    backbone_lr: float = 2e-5
    head_lr: float = 1e-3
    head_only_epochs: int = 5

    seed: int = 42

    def __post_init__(self):
        total = self.train_ratio + self.val_ratio + self.test_ratio
        assert abs(total - 1.0) < 1e-6, f"Split ratios must sum to 1, got {total}"

    @property
    def num_classes(self):
        return len(self.class_names)

    def subdir(self, name):
        p = Path(self.project_dir) / name
        p.mkdir(parents=True, exist_ok=True)
        return p

    def save(self):
        path = self.subdir("configs") / "config.json"
        path.write_text(json.dumps(asdict(self), indent=2))
        return path

cfg = Config()
for d in ["data_raw", "splits", "cache", "models", "results", "figures", "reports", "configs"]:
    cfg.subdir(d)
print("Config saved to:", cfg.save())
print("Classes:", cfg.class_names, "->", cfg.num_classes, "outputs")

**Step 03 — Reproducibility and device.** Seed Python, NumPy, and PyTorch; select CUDA when available.

In [ ]:
# Reproducibility and device
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # same result every run
    torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(cfg.seed)
device = get_device()
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

**Step 04 — Record library versions.** Print the Python and main package versions for reproducibility.

In [ ]:
# Record library versions
import sys, torch, torchvision, timm, sklearn, pandas
import gradio
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("timm       :", timm.__version__)
print("sklearn    :", sklearn.__version__)
print("pandas     :", pandas.__version__)
print("Gradio     :", gradio.__version__)

**Step 05 — Extract the dataset.** Unpack the downloaded NASTaR archive into the local data directory.

In [ ]:
# Extract the dataset
import zipfile
from tqdm import tqdm

def unzip_all(src_dir, dst_dir):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    zips = sorted(src_dir.glob("*.zip"))
    if not zips:
        print("No .zip files found in", src_dir)
        return
    for z in zips:
        print("Extracting", z.name)
        with zipfile.ZipFile(z) as zf:
            for member in tqdm(zf.namelist()):
                zf.extract(member, dst_dir)

src_dir = r"https://drive.google.com/drive/folders/1wLNCKsSnHO9T_s42D65pyx4tU9RCtYWK?usp=sharing"
unzip_all(src_dir, cfg.local_data_dir)

**Step 06 — Inspect the dataset layout.** Locate scene folders, ship patches, and metadata files before indexing.

In [ ]:
# Inspect the dataset layout.
from collections import Counter

def show_tree(root, max_depth=3, max_children=5):
    root = Path(root)
    def _walk(path, depth):
        if depth > max_depth:
            return
        children = sorted(path.iterdir())
        dirs = [c for c in children if c.is_dir()]
        files = [c for c in children if c.is_file()]
        indent = "    " * depth
        for d in dirs[:max_children]:
            print(f"{indent}{d.name}/")
            _walk(d, depth + 1)
        if len(dirs) > max_children:
            print(f"{indent}... (+{len(dirs) - max_children} more folders)")
        for f in files[:max_children]:
            print(f"{indent}{f.name}")
        if len(files) > max_children:
            print(f"{indent}... (+{len(files) - max_children} more files)")
    print(root.name + "/")
    _walk(root, 1)

root = Path(cfg.local_data_dir)

print("=== Folder tree ===")
show_tree(root)

print("\n=== File types ===")
print(Counter(p.suffix.lower() for p in root.rglob("*") if p.is_file()))

print("\n=== Possible metadata files ===")
meta_ext = {".csv", ".xlsx", ".xls", ".json", ".txt", ".md"}
meta_files = [p for p in root.rglob("*") if p.suffix.lower() in meta_ext]
for p in meta_files[:10]:
    print(p.relative_to(root))
print("total:", len(meta_files))

**Step 07 — Compare patch formats.** Inspect a raw TIFF and its uint8 visualization copy; use the raw data for training.

In [ ]:
# Compare patch formats.
import numpy as np
import tifffile
from PIL import Image

IMAGE_EXTENSIONS = {
    ".tif", ".tiff", ".png", ".jpg", ".jpeg",
    ".bmp", ".webp"
}

def peek_image(path):
    try:
        try:
            img = tifffile.imread(path)
        except Exception:
            img = np.array(Image.open(path))

        print(f"  file   : {path.name}")
        print(f"  shape  : {img.shape}")
        print(f"  dtype  : {img.dtype}")
        print(f"  min/max: {img.min()} / {img.max()}")
        print(f"  mean   : {img.mean():.2f}")

    except Exception as e:
        print(f"  Could not read {path.name}: {e}")

for name in ("ship_patches", "ship_patches_uint8"):
    d = next(
        (p for p in root.rglob(name) if p.is_dir()),
        None
    )

    print(f"\n[{name}]")

    if d is None:
        print("  Folder not found")
        continue

    files = sorted(
        p for p in d.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not files:
        print("  No supported image files found")
        continue

    peek_image(files[0])

**Step 08 — Inspect AIS metadata.** Check the AIS columns and label fields used to build the master index.

In [ ]:
# Inspect AIS metadata
import pandas as pd

root = Path(cfg.local_data_dir)
ais_files = sorted(root.rglob("AIS.csv"))
print("AIS.csv files:", len(ais_files))

df0 = pd.read_csv(ais_files[0], encoding_errors="replace")
print("\nShape of first file:", df0.shape)
print("\nColumns:", list(df0.columns))
display(df0.head(3))

frames = [pd.read_csv(f, encoding_errors="replace") for f in ais_files]
all_ais = pd.concat(frames, ignore_index=True)
print("\nTotal rows across all AIS.csv:", len(all_ais))

for col in all_ais.columns:
    if any(k in col.lower() for k in ("ship type", "quality", "shoreline")):
        print(f"\n--- {col} ---")
        print(all_ais[col].value_counts(dropna=False).head(15))

## Section 02 : DATASET INDEXING
Create one master row per image patch and join it to its AIS metadata.

**Step 01 — Locate scenes.** Check scene folders and the required AIS and patch directories.

In [ ]:
# Locate scenes
from pathlib import Path
import re
import numpy as np
import pandas as pd

root = Path(cfg.local_data_dir)
data_dir = next(p for p in root.rglob("Data") if p.is_dir())
scene_dirs = sorted(p for p in data_dir.iterdir()
                    if p.is_dir() and p.name.startswith("NovaSAR"))

print("Data folder  :", data_dir)
print("Scene folders:", len(scene_dirs))
no_ais  = [p.name for p in scene_dirs if not (p / "AIS.csv").exists()]
no_ship = [p.name for p in scene_dirs if not (p / "ship_patches").exists()]
print("Scenes without AIS.csv     :", len(no_ais), no_ais)
print("Scenes without ship_patches:", len(no_ship))
print("Stray .zip files:", [str(p.relative_to(root)) for p in root.rglob("*.zip")][:5])

**Step 02 — Index image files.** Extract stable patch keys and store each TIFF's scene and relative path.

In [ ]:
# Index image files.
PATCH_KEY = re.compile(r"^(.*?_patch_\d+)")

def patch_key(stem: str) -> str:
    """Scene name + patch number, ignoring the ship-type suffix."""
    m = PATCH_KEY.match(stem)
    return m.group(1) if m else stem

rows = []
for sd in scene_dirs:
    folder = sd / "ship_patches"
    if folder.exists():
        for p in sorted(folder.glob("*.tif*")):
            rows.append({"scene": sd.name,
                         "file_stem": p.stem,
                         "patch_key": patch_key(p.stem),
                         "rel_path": str(p.relative_to(data_dir))})

files_df = pd.DataFrame(rows)
print("ship_patches files:", len(files_df))
print("Duplicate (scene, patch_key):",
      files_df.duplicated(["scene", "patch_key"]).sum(), "(should be 0)")

**Step 02b — Validate patch keys.** Check that filename parsing produces usable, unique identifiers.

In [ ]:
# @titleValidate patch keys.
import zipfile

for sd in scene_dirs:
    z = sd / "AIS.zip"
    if z.exists() and not (sd / "AIS.csv").exists():
        with zipfile.ZipFile(z) as zf:
            print(sd.name, "->", zf.namelist())
            zf.extractall(sd)

print("Scenes still without AIS.csv:",
      [p.name for p in scene_dirs if not (p / "AIS.csv").exists()])

**Step 03 — Combine AIS files.** Standardize metadata columns and stack the scene-level AIS tables.

In [ ]:
# Combine AIS files
def clean_col(c):
    return re.sub(r"\s+", "_", c.strip().lstrip("#").strip().lower())

frames = []
for sd in scene_dirs:
    f = sd / "AIS.csv"
    if f.exists():
        d = pd.read_csv(f, encoding_errors="replace")
        d.columns = [clean_col(c) for c in d.columns]
        d["scene"] = sd.name
        frames.append(d)

ais = pd.concat(frames, ignore_index=True)
ais["ship_type"] = ais["ship_type"].astype(str).str.strip()
ais["patch_key"] = (ais["patch_name"].astype(str).str.strip()
                    .map(lambda s: patch_key(re.sub(r"\.tiff?$", "", s, flags=re.I))))

print("AIS rows:", len(ais))
print("Duplicate (scene, patch_key) in AIS:",
      ais.duplicated(["scene", "patch_key"]).sum())

**Step 04 — Join and validate.** Match patches to AIS rows and check filename labels against AIS ship types.

In [ ]:
# Join and validate
master = ais.merge(files_df, on=["scene", "patch_key"], how="left", indicator=True)
has_file = master["_merge"] == "both"
print("AIS rows with a patch file:", has_file.sum(), "/", len(master))

orph = files_df.merge(ais[["scene", "patch_key"]].drop_duplicates(),
                      on=["scene", "patch_key"], how="left", indicator=True)
orph = orph[orph["_merge"] == "left_only"]
print("Patch files with no AIS row:", len(orph))

# Independent check: the file name ends with the ship type
def name_label(stem):
    m = re.search(r"_patch_\d+_(.+)$", str(stem))
    return m.group(1) if m else None

norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
chk = master[has_file].copy()
agree = chk["file_stem"].map(name_label).map(norm) == chk["ship_type"].map(norm)
print(f"File-name label == AIS label: {agree.mean()*100:.1f}%  ({(~agree).sum()} mismatches)")
display(chk.loc[~agree, ["scene", "file_stem", "ship_type"]].head())

**Step 05 — Save the master index.** Keep matched rows, record uint8 paths and quality groups, then write `master_index.csv`.

In [ ]:
#  Save the master index
master = master[has_file].drop(columns="_merge").reset_index(drop=True)

def uint8_rel(rel):
    p = Path(rel)
    return str(p.parent.parent / "ship_patches_uint8" / f"{p.stem}_uint8{p.suffix}")

master["rel_path_uint8"] = master["rel_path"].map(uint8_rel)
master["uint8_exists"] = [(data_dir / r).exists() for r in master["rel_path_uint8"]]
print(f"uint8 twin found: {master['uint8_exists'].mean()*100:.1f}%")

def qgroup(q):
    if pd.isna(q):
        return "blank"
    return {1.0: "1", 0.0: "0", 9.0: "9"}.get(float(q), "other")

master["quality_group"] = master["quality"].map(qgroup)

out = cfg.subdir("splits") / "master_index.csv"
master.to_csv(out, index=False)
print("Saved:", out, master.shape)

**Step 06 — Check quality and leakage.** Inspect quality/shoreline patterns and repeated vessel identifiers before filtering.

In [ ]:
# Check quality and leakage.
print("=== Quality group x shoreline ===")
print(pd.crosstab(master["quality_group"], master["shoreline"], margins=True))

four = cfg.class_names
type_view = master["ship_type"].where(master["ship_type"].isin(four), "(other types)")
print("\n=== Ship type x quality group ===")
print(pd.crosstab(type_view, master["quality_group"], margins=True))

per_mmsi = master.groupby("mmsi").size()
print("\nUnique MMSI:", master["mmsi"].nunique())
print("MMSI with more than one patch:", (per_mmsi > 1).sum(),
      "| max patches for one MMSI:", per_mmsi.max())

**Step 07 — Check TIFF dtype and shape.** Sample raw patches to verify dimensions, numeric type, and value range.

In [ ]:
# Check TIFF dtype and shape
import tifffile

sample = master.sample(60, random_state=cfg.seed)
recs = []
for rel in sample["rel_path"]:
    a = tifffile.imread(data_dir / rel)
    recs.append({"dtype": str(a.dtype), "shape": a.shape,
                 "min": a.min(), "max": a.max(),
                 "p50": np.percentile(a, 50), "p99": np.percentile(a, 99)})
s = pd.DataFrame(recs)
print(s["dtype"].value_counts())
print(s["shape"].value_counts())
print(s[["min", "max", "p50", "p99"]].describe().round(1))

## Section 03 : DATASET ANALYSIS
Generate the dataset counts and plots used to justify the four-class subset and preprocessing choices.

**Step 01 — Build the working subset.** Keep the selected quality groups and Cargo, Tanker, Fishing, and Passenger labels; report counts.

In [ ]:
# Build the working subset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import tifffile

master = pd.read_csv(cfg.subdir("splits") / "master_index.csv")
data_dir = next(p for p in Path(cfg.local_data_dir).rglob("Data") if p.is_dir())

kept = master["quality"].isna() | (master["quality"] == 1)
print("Kept after quality filter:", kept.sum(), "(paper: 1891)")

df4 = master[kept & master["ship_type"].isin(cfg.class_names)].reset_index(drop=True)
print("Four-class subset:", len(df4))

counts = df4["ship_type"].value_counts().loc[cfg.class_names]
ct = pd.crosstab(df4["ship_type"], df4["shoreline"]).loc[cfg.class_names]
print("\n", ct)
print("\n% inshore / offshore within each class:")
print((ct.div(ct.sum(axis=1), axis=0) * 100).round(1))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(counts.index, counts.values)
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 5, str(v), ha="center")
ax[0].set_title("Images per class (quality-filtered)")
ct.plot(kind="bar", stacked=True, ax=ax[1], rot=0)
ax[1].set_title("Inshore vs offshore per class")
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_class_distribution.png", dpi=150)
plt.show()

**Step 02 — View image samples.** Display representative uint8 images from each target class.

In [ ]:
# @title
def show_samples(df, n=6, seed=cfg.seed):
    fig, axes = plt.subplots(len(cfg.class_names), n,
                             figsize=(2.2 * n, 2.4 * len(cfg.class_names)))
    for r, cls in enumerate(cfg.class_names):
        rows = df[df["ship_type"] == cls].sample(n, random_state=seed)
        for c, (_, row) in enumerate(rows.iterrows()):
            img = tifffile.imread(data_dir / row["rel_path_uint8"])
            axes[r, c].imshow(img, cmap="gray", vmin=0, vmax=255)
            axes[r, c].axis("off")
            if c == 0:
                axes[r, c].set_title(cls, loc="left", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_samples.png", dpi=150)
    plt.show()

show_samples(df4)

**Step 03 — Inspect intensity statistics.** Measure raw-image brightness, contrast, and background characteristics.

In [ ]:
# @title
def patch_stats(path):
    a = tifffile.imread(path).astype(np.float32)
    h, w = a.shape[:2]
    cy, cx = h // 2, w // 2
    center = a[cy - 48:cy + 48, cx - 48:cx + 48]            # 96x96 window around the ship
    bg = np.median(np.concatenate([a[:64].ravel(), a[-64:].ravel()]))  # top+bottom strips
    c99 = np.percentile(center, 99)
    return {"median": np.median(a), "p99": np.percentile(a, 99), "max": a.max(),
            "bg_median": bg, "center_p99": c99,
            "contrast_log10": np.log10(max(c99, 1) / max(bg, 1)),
            "frac_zero": (a == 0).mean()}

stats = pd.DataFrame([patch_stats(data_dir / p) for p in tqdm(df4["rel_path"])])
df4s = pd.concat([df4, stats], axis=1)
df4s.to_csv(cfg.subdir("results") / "patch_stats_4class.csv", index=False)

cols = ["bg_median", "p99", "contrast_log10"]
print(df4s.groupby("ship_type")[cols + ["frac_zero"]].median().round(3).loc[cfg.class_names])

titles = ["Background level (median)", "Brightness (99th percentile)", "Ship contrast (log10 ratio)"]
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a_, col, t in zip(ax, cols, titles):
    data = [df4s.loc[df4s["ship_type"] == c, col] for c in cfg.class_names]
    a_.boxplot(data, showfliers=False)
    a_.set_xticks(range(1, len(cfg.class_names) + 1), cfg.class_names)
    a_.set_title(t)
    if col != "contrast_log10":
        a_.set_yscale("log")
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_intensity_stats.png", dpi=150)
plt.show()

**Step 04 — Inspect ship dimensions.** Summarize AIS length/width and evaluate a metadata-only baseline.

In [ ]:
# @title
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

size = df4[(df4["length"] > 0) & (df4["width"] > 0)].copy()
print(f"Rows with valid length/width: {len(size)} / {len(df4)}")
print(size.groupby("ship_type")[["length", "width"]].median().loc[cfg.class_names])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a_, col in zip(ax, ["length", "width"]):
    a_.boxplot([size.loc[size["ship_type"] == c, col] for c in cfg.class_names], showfliers=False)
    a_.set_xticks(range(1, len(cfg.class_names) + 1), cfg.class_names)
    a_.set_title(f"{col} (m), from AIS")
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_ship_size.png", dpi=150)
plt.show()

# Baseline that sees ONLY length and width, no image
X = size[["length", "width"]].values
y = size["ship_type"].values
g = size["mmsi"].fillna(-1).astype(int).values
scores = []
for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=cfg.seed).split(X, y, g):
    clf = DecisionTreeClassifier(max_depth=3, class_weight="balanced",
                                 random_state=cfg.seed).fit(X[tr], y[tr])
    p = clf.predict(X[te])
    scores.append((balanced_accuracy_score(y[te], p), f1_score(y[te], p, average="macro")))
sc = np.array(scores)
print(f"\nLength+width only: balanced acc {sc[:, 0].mean():.3f}, "
      f"macro-F1 {sc[:, 1].mean():.3f} (+/- {sc[:, 1].std():.3f}); chance = 0.250")

**Step 05 — Inspect vessel and scene groups.** Count unique MMSIs, repeated vessels, and scene groups to support split design.

In [ ]:
# @title
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

per_mmsi = df4.groupby("mmsi").size()
per_scene = df4.groupby("scene").size()
print("Unique vessels (MMSI):", len(per_mmsi))
print("Vessels with >1 patch:", (per_mmsi > 1).sum(), "| max patches for one vessel:", per_mmsi.max())
print("Vessels with conflicting labels:", (df4.groupby("mmsi")["ship_type"].nunique() > 1).sum())
print(f"Scenes: {len(per_scene)} | patches per scene: mean {per_scene.mean():.2f}, max {per_scene.max()}")

def component_sizes(df, keys):
    n, r, c = len(df), [], []
    for k in keys:
        first = {}
        for i, v in enumerate(df[k].values):
            if v in first:
                r.append(first[v]); c.append(i)
            else:
                first[v] = i
    adj = coo_matrix((np.ones(len(r)), (r, c)), shape=(n, n))
    _, lab = connected_components(adj, directed=False)
    return np.bincount(lab)

for keys in (["mmsi"], ["scene"], ["mmsi", "scene"]):
    s = component_sizes(df4, keys)
    print(f"Group by {keys}: {len(s)} groups, largest = {s.max()} patches "
          f"({s.max() / len(df4) * 100:.1f}% of data)")

# FILTERING, PREPROCESSING & SPLITTING

## Section 04 : Curation and splitting
Create the final four-class table and group-aware development and cross-validation splits.

**Step 01 — Build the final table.** Apply the quality and class filters and assign numeric labels.

In [ ]:
# @title
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

master = pd.read_csv(cfg.subdir("splits") / "master_index.csv")
kept = master["quality"].isna() | (master["quality"] == 1)
df = master[kept & master["ship_type"].isin(cfg.class_names)].reset_index(drop=True)

class_to_idx = {c: i for i, c in enumerate(cfg.class_names)}
df["label"] = df["ship_type"].map(class_to_idx)
df["stratum"] = df["ship_type"] + "|" + df["shoreline"]       # 8 strata

df["mmsi_group"] = df["mmsi"].astype("Int64").astype(str)
miss = df["mmsi"].isna()
df.loc[miss, "mmsi_group"] = [f"missing_{i}" for i in df.index[miss]]   # unknown vessel = own group
df["scene_group"] = df["scene"]

print(len(df), "images |", df["mmsi_group"].nunique(), "vessel groups |",
      df["scene_group"].nunique(), "scene groups")
print(df["stratum"].value_counts())
(cfg.subdir("splits") / "class_map.json").write_text(json.dumps(class_to_idx, indent=2))

**Step 02 — Define grouped splits.** Keep all patches from one MMSI or scene within a single split/fold.

In [ ]:
# @title
def sgkf(n_splits, seed):
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

def holdout_split(df, group_col, test_ratio, val_ratio, seed):
    """Train/val/test split that is stratified by class x shoreline and never
    splits one group (vessel or scene) across subsets. Ratios are approximate."""
    n_test = round(1 / test_ratio)
    rest_idx, test_idx = next(iter(sgkf(n_test, seed).split(
        df, df["stratum"].values, df[group_col].values)))
    rest = df.iloc[rest_idx]
    n_val = round(1 / (val_ratio / (1 - test_ratio)))
    _, va_rel = next(iter(sgkf(n_val, seed).split(
        rest, rest["stratum"].values, rest[group_col].values)))
    split = np.array(["train"] * len(df), dtype=object)
    split[rest_idx[va_rel]] = "val"
    split[test_idx] = "test"
    return split

def cv_folds(df, group_col, n_folds, seed):
    fold = np.full(len(df), -1)
    for k, (_, te) in enumerate(sgkf(n_folds, seed).split(
            df, df["stratum"].values, df[group_col].values)):
        fold[te] = k
    return fold

def fold_split(df, fold_col, k, n_folds):
    """For test fold k: the next fold is validation, the rest is training."""
    return np.where(df[fold_col] == k, "test",
           np.where(df[fold_col] == (k + 1) % n_folds, "val", "train"))

df["holdout"]    = holdout_split(df, "mmsi_group", cfg.test_ratio, cfg.val_ratio, cfg.seed)
df["fold_mmsi"]  = cv_folds(df, "mmsi_group",  cfg.n_folds, cfg.seed)
df["fold_scene"] = cv_folds(df, "scene_group", cfg.n_folds, cfg.seed)

**Step 03 — Validate splits.** Check class/shoreline balance and confirm no group appears across subsets.

In [ ]:
# @title
print("=== Development holdout (vessel-grouped) ===")
print((df["holdout"].value_counts(normalize=True) * 100).round(1))
print(pd.crosstab(df["ship_type"], df["holdout"]).loc[cfg.class_names, ["train", "val", "test"]])
print(pd.crosstab(df["shoreline"], df["holdout"])[["train", "val", "test"]])

def leaks(col, group_col):
    return int((df.groupby(group_col)[col].nunique() > 1).sum())

l1, l2, l3 = (leaks("holdout", "mmsi_group"),
              leaks("fold_mmsi", "mmsi_group"),
              leaks("fold_scene", "scene_group"))
print("\nVessels split across holdout subsets :", l1, "(must be 0)")
print("Vessels split across CV folds        :", l2, "(must be 0)")
print("Scenes split across scene-CV folds   :", l3, "(must be 0)")
assert l1 == 0 and l2 == 0 and l3 == 0, "Leakage detected, do not continue"

train_scenes = set(df.loc[df["holdout"] == "train", "scene_group"])
test = df[df["holdout"] == "test"]
print(f"\nTest patches that share a scene with a training patch: "
      f"{test['scene_group'].isin(train_scenes).mean() * 100:.1f}%")

print("\n=== Vessel-grouped CV: images per class per fold ===")
print(pd.crosstab(df["ship_type"], df["fold_mmsi"]).loc[cfg.class_names])
print("\n=== Scene-grouped CV: images per class per fold ===")
print(pd.crosstab(df["ship_type"], df["fold_scene"]).loc[cfg.class_names])

**Step 04 — Save split assignments.** Write the row-level holdout and grouped-fold assignments to `splits.csv`.

In [ ]:
# @title
cols = ["rel_path", "rel_path_uint8", "ship_type", "label", "shoreline",
        "mmsi_group", "scene_group", "holdout", "fold_mmsi", "fold_scene",
        "length", "width"]
out = cfg.subdir("splits") / "splits.csv"
df[cols].to_csv(out, index=False)
print("Saved:", out, df[cols].shape)

In [ ]:
# @title
# load the split table saved by Section 04
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SPLITS_CSV = cfg.subdir("splits") / "splits.csv"
if not SPLITS_CSV.exists():
    raise FileNotFoundError(f"{SPLITS_CSV} not found. Run Section 04 (Curation and splitting) first.")

df = pd.read_csv(SPLITS_CSV)
required = ["rel_path", "ship_type", "label", "shoreline", "mmsi_group",
            "scene_group", "holdout", "fold_mmsi", "fold_scene"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"splits.csv is missing columns: {missing_cols}"
assert df["ship_type"].isin(cfg.class_names).all(), "Unexpected classes in splits.csv"
class_to_idx = {c: i for i, c in enumerate(cfg.class_names)}
assert (df["label"] == df["ship_type"].map(class_to_idx)).all(), "label column does not follow cfg.class_names order"

N = len(df)
REL_HASH = hashlib.md5("\n".join(df["rel_path"]).encode()).hexdigest()
CACHE_DIR = cfg.subdir("cache")
print(f"{N} images | table fingerprint {REL_HASH[:8]}")
print(df["ship_type"].value_counts().loc[cfg.class_names].to_string())

## Section 05 : Preprocessing and caching
Build reusable raw and model-input caches. The saved sample check currently reports `uint16` raw TIFFs; the dataset landing page describes float32 GeoTIFF backscatter, so verify the downloaded release's `readme.txt`, `LICENSE.txt`, and TIFF metadata before final submission.

**Step 01 — Make raw patches available.** Reuse local files or extract only the patches listed in `splits.csv`.

In [ ]:
# @title
import zipfile
import tifffile
from tqdm.auto import tqdm

RAW_NPY  = CACHE_DIR / "raw_uint16.npy"
RAW_META = CACHE_DIR / "raw_uint16_meta.json"

def raw_cache_ok():
    if not (RAW_NPY.exists() and RAW_META.exists()):
        return False
    meta = json.loads(RAW_META.read_text())
    return meta.get("rel_hash") == REL_HASH and meta.get("n") == N

def find_data_dir():
    root = Path(cfg.local_data_dir)
    if not root.exists():
        return None
    return next((p for p in root.glob("**/Data") if p.is_dir()), None)

def extract_needed(zip_path, dst_dir, needed_rel):
    """Extract only the patches listed in splits.csv."""
    needed = set(needed_rel)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        todo = []
        for m in zf.infolist():
            if m.is_dir():
                continue
            k = m.filename.find("Data/")
            if k == -1 or (k > 0 and m.filename[k - 1] != "/"):
                continue
            if m.filename[k + 5:] in needed:
                todo.append(m)
        print(f"Extracting {len(todo)} of {len(needed)} needed patches from {Path(zip_path).name}")
        for m in tqdm(todo):
            zf.extract(m, dst_dir)

if raw_cache_ok():
    print("Raw cache already built on Drive: TIFF extraction is not needed in this session.")
else:
    data_dir = find_data_dir()
    have_all = data_dir is not None and all((data_dir / r).exists() for r in df["rel_path"])
    if have_all:
        print("All patches already on the local disk:", data_dir)
    else:
        zips = sorted(cfg.subdir("data_raw").glob("*.zip"))
        if not zips:
            raise FileNotFoundError("No dataset zip found in data_raw/. Download it first (Section 01).")
        extract_needed(zips[0], cfg.local_data_dir, df["rel_path"])
        data_dir = find_data_dir()
        missing = [r for r in df["rel_path"] if not (data_dir / r).exists()]
        print("Patches still missing after extraction:", len(missing))
        assert not missing, "Some patches could not be extracted"

**Step 02 — Build the raw cache.** Store the indexed 512×512 patches in row order in `raw_uint16.npy`.

In [ ]:
# @title
def build_raw_cache(data_dir):
    raw = np.empty((N, 512, 512), dtype=np.uint16)
    n_cast = 0
    for i, rel in enumerate(tqdm(df["rel_path"], desc="reading TIFFs")):
        a = tifffile.imread(data_dir / rel)
        if a.shape != (512, 512):
            raise ValueError(f"{rel}: unexpected shape {a.shape}")
        if a.dtype != np.uint16:
            n_cast += 1
            a = np.clip(a, 0, 65535)
        raw[i] = a
    np.save(RAW_NPY, raw)
    RAW_META.write_text(json.dumps({"rel_hash": REL_HASH, "n": N, "shape": list(raw.shape),
                                    "dtype": "uint16", "n_cast": n_cast}))
    print(f"Saved {RAW_NPY.name}: {raw.shape}, {raw.nbytes / 1e6:.0f} MB, {n_cast} patches needed casting")

_RAW = {}
def get_raw():
    """Load the raw uint16 array (memoised)."""
    if "raw" not in _RAW:
        if not raw_cache_ok():
            raise FileNotFoundError("Raw cache missing or stale: re-run this cell.")
        _RAW["raw"] = np.load(RAW_NPY)
    return _RAW["raw"]

if not raw_cache_ok():
    build_raw_cache(find_data_dir())
raw = get_raw()
print("Raw cache:", raw.shape, raw.dtype, f"{raw.nbytes / 1e6:.0f} MB")

sub = raw[:, ::8, ::8].reshape(N, -1).astype(np.float32)
summ = pd.DataFrame({"ship_type": df["ship_type"],
                     "bg_median": np.median(sub, axis=1),
                     "p99": np.percentile(sub, 99, axis=1),
                     "max": raw.reshape(N, -1).max(axis=1)})
print("\nPer-class medians (raw digital numbers):")
print(summ.groupby("ship_type").median().round(1).loc[cfg.class_names])

**Step 03 — Define preprocessing.** Implement resizing/cropping, intensity scaling (`log`, `percentile`, `bgnorm`), and model normalization.

In [ ]:
# @title
import cv2
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

class Preprocessor:
    """Class-diagram Preprocessor."""

    LOG_LO, LOG_HI = float(np.log1p(10.0)), float(np.log1p(10000.0))   # fixed range for "log"
    BG_SCALE = 6.0                                                    # natural-log units for "bgnorm"
    MODES = ("log", "percentile", "bgnorm")
    CROPS = ("resize", "center_crop")

    def __init__(self, target_size=256, crop_mode="resize", intensity_mode="log",
                 aug_enabled=True, input_size=224):
        assert intensity_mode in self.MODES, f"intensity_mode must be one of {self.MODES}"
        assert crop_mode in self.CROPS, f"crop_mode must be one of {self.CROPS}"
        assert input_size <= target_size, "input_size cannot exceed target_size"
        self.target_size, self.crop_mode, self.intensity_mode = target_size, crop_mode, intensity_mode
        self.aug_enabled, self.input_size = aug_enabled, input_size
        self.norm_mean, self.norm_std = 0.0, 1.0
        self._affine = T.RandomAffine(degrees=10, translate=(0.04, 0.04),
                                      interpolation=InterpolationMode.BILINEAR)

    @classmethod
    def from_cfg(cls, cfg, **overrides):
        args = dict(target_size=cfg.target_size, crop_mode=cfg.crop_mode,
                    intensity_mode=cfg.intensity_mode, aug_enabled=cfg.aug_enabled,
                    input_size=cfg.swin_input_size)
        args.update(overrides)
        return cls(**args)

    @property
    def tag(self):
        return f"{self.intensity_mode}_{self.crop_mode}_{self.target_size}"

    def recipe(self):
        """Settings that define the cached array (stored next to it)."""
        return {"target_size": self.target_size, "crop_mode": self.crop_mode,
                "intensity_mode": self.intensity_mode, "log_lo": self.LOG_LO,
                "log_hi": self.LOG_HI, "bg_scale": self.BG_SCALE}

    # ---------- deterministic steps (cached) ----------
    def resize(self, img):
        s = self.target_size
        if self.crop_mode == "resize":
            return cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA)
        h, w = img.shape
        t, l = (h - s) // 2, (w - s) // 2
        return img[t:t + s, l:l + s].copy()

    def scale_intensity(self, img):
        y = np.log1p(img)
        if self.intensity_mode == "log":
            y = (y - self.LOG_LO) / (self.LOG_HI - self.LOG_LO)
            return np.clip(y, 0.0, 1.0).astype(np.float32)
        if self.intensity_mode == "percentile":
            lo, hi = np.percentile(y, (1.0, 99.5))
            y = (y - lo) / (hi - lo + 1e-6)
            return np.clip(y, 0.0, 1.0).astype(np.float32)
        y = (y - np.median(y)) / self.BG_SCALE                      # "bgnorm"
        return np.clip(y, -0.5, 1.0).astype(np.float32)

    def preprocess(self, raw_img):
        return self.scale_intensity(self.resize(np.asarray(raw_img, dtype=np.float32)))

    def cache_dataset(self, raw):
        out = np.empty((len(raw), self.target_size, self.target_size), dtype=np.float16)
        for i in tqdm(range(len(raw)), desc=f"caching {self.tag}"):
            out[i] = self.preprocess(raw[i])
        return out

    # ---------- statistics (training rows only) ----------
    def fit_normalization(self, X, idx):
        xi = X[idx].astype(np.float32)
        self.norm_mean, self.norm_std = float(xi.mean()), float(xi.std() + 1e-8)
        return self.norm_mean, self.norm_std

    def normalize(self, x):
        return (x - self.norm_mean) / self.norm_std

    # ---------- training-time only ----------
    def augment(self, x):
        """x: float tensor (1, H, W), already normalised (so a fill value of 0 equals the mean level)."""
        x = torch.rot90(x, random.randint(0, 3), (1, 2))
        if random.random() < 0.5:
            x = torch.flip(x, (2,))
        x = self._affine(x)
        if random.random() < 0.3:
            x = x + 0.05 * torch.randn_like(x)
        return x

    def crop_to_input(self, x, train):
        h, s = x.shape[-1], self.input_size
        if s >= h:
            return x
        if train:
            i, j = random.randint(0, h - s), random.randint(0, h - s)
        else:
            i = j = (h - s) // 2
        return x[..., i:i + s, j:j + s]

    def state(self):
        """Everything the app needs to repeat this preprocessing (saved in the model card)."""
        return {**self.recipe(), "input_size": self.input_size, "aug_enabled": self.aug_enabled,"norm_mean": self.norm_mean, "norm_std": self.norm_std}

prep = Preprocessor.from_cfg(cfg)
print("Preprocessor ready:", prep.tag, "| model input", prep.input_size)

**Step 04 — Build the feature cache.** Create or load the float16 array for the selected preprocessing recipe.

In [ ]:
# @title
import time

_CACHE_MEMO = {}

def cache_paths(p):
    stem = f"X_{p.tag}"
    return CACHE_DIR / f"{stem}.npy", CACHE_DIR / f"{stem}.json"

def get_cache(p, force=False):
    """Return the cached float16 array for this preprocessing recipe, building it if needed."""
    if p.tag in _CACHE_MEMO and not force:
        return _CACHE_MEMO[p.tag]
    npy, meta_path = cache_paths(p)
    if npy.exists() and meta_path.exists() and not force:
        meta = json.loads(meta_path.read_text())
        if meta["rel_hash"] == REL_HASH and meta["n"] == N and meta["recipe"] == p.recipe():
            X = np.load(npy)
            print(f"[cache] loaded {npy.name} {X.shape} {X.dtype}")
            _CACHE_MEMO[p.tag] = X
            return X
        print("[cache] stale, rebuilding:", npy.name)
    X = p.cache_dataset(get_raw())
    np.save(npy, X)
    meta_path.write_text(json.dumps({"rel_hash": REL_HASH, "n": N, "recipe": p.recipe(),
                                     "created": time.strftime("%Y-%m-%d %H:%M:%S")}))
    print(f"[cache] built {npy.name} {X.shape} ({X.nbytes / 1e6:.0f} MB)")
    _CACHE_MEMO[p.tag] = X
    return X

X = get_cache(prep)
Xf = X.astype(np.float32)
print(f"\nShape {X.shape} | dtype {X.dtype} | {X.nbytes / 1e6:.0f} MB")
print(f"Value range {Xf.min():.3f} to {Xf.max():.3f} | mean {Xf.mean():.3f} | std {Xf.std():.3f}")
print("Any NaN or Inf:", bool(~np.isfinite(Xf).all()))
del Xf

**Step 05 — Compare intensity recipes.** Visualize each recipe and inspect how it changes the sea-background level.

In [ ]:
# @title
modes = Preprocessor.MODES
rng = np.random.default_rng(cfg.seed)
raw = get_raw()

fig, axes = plt.subplots(cfg.num_classes, len(modes) + 1, figsize=(3.0 * (len(modes) + 1), 3.0 * cfg.num_classes))
for r, c in enumerate(cfg.class_names):
    i = int(rng.choice(np.where(df["ship_type"].values == c)[0]))
    axes[r, 0].imshow(np.log1p(raw[i].astype(np.float32)), cmap="gray")
    axes[r, 0].set_title(f"{c}: raw (log view)", fontsize=9)
    for k, m in enumerate(modes):
        pm = Preprocessor.from_cfg(cfg, intensity_mode=m)
        axes[r, k + 1].imshow(pm.preprocess(raw[i]), cmap="gray")
        axes[r, k + 1].set_title(m, fontsize=9)
for a in axes.ravel():
    a.axis("off")
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_intensity_recipes.png", dpi=130)
plt.show()

# background level of each image under each recipe (resize once, then scale three ways)
base = Preprocessor.from_cfg(cfg)
R = np.stack([base.resize(raw[i].astype(np.float32)) for i in range(N)])
rows = {}
for m in modes:
    pm = Preprocessor.from_cfg(cfg, intensity_mode=m)
    med = np.empty(N)
    s1 = s2 = 0.0
    cnt = 0
    for i in range(N):
        z = pm.scale_intensity(R[i])
        med[i] = np.median(z)
        s1 += float(z.sum(dtype=np.float64))
        s2 += float((z.astype(np.float64) ** 2).sum())
        cnt += z.size
    pix_std = float(np.sqrt(max(s2 / cnt - (s1 / cnt) ** 2, 1e-12)))
    cls_med = pd.Series(med).groupby(df["ship_type"].values).median().loc[cfg.class_names]
    rows[m] = {**cls_med.round(3).to_dict(),
               "background cue (input std)": round(float((cls_med.max() - cls_med.min()) / pix_std), 2)}
del R
print("Median background level per class under each recipe (and the cue a model could exploit):")
print(pd.DataFrame(rows).to_string())

**Step 06 — Optional caches.** Enable this only when running the preprocessing ablation experiments.

In [ ]:
# @title
BUILD_ABLATION_CACHES = False

if BUILD_ABLATION_CACHES:
    for kw in ({"intensity_mode": "percentile"}, {"intensity_mode": "bgnorm"}, {"crop_mode": "center_crop"}):
        get_cache(Preprocessor.from_cfg(cfg, **kw))
print("Caches on Drive:", sorted(p.name for p in CACHE_DIR.glob("X_*.npy")))

## Section 06 : Dataset and DataLoaders
Prepare leakage-safe batches with training-only normalization and training-only augmentation.

**Step 01 — Select and verify a split.** Build train, validation, and test indices and assert group separation.

In [ ]:
# @title
SCHEMES = ("holdout", "cv_mmsi", "cv_scene")

def fold_split(frame, fold_col, k, n_folds):
    return np.where(frame[fold_col] == k, "test",
           np.where(frame[fold_col] == (k + 1) % n_folds, "val", "train"))

def get_split(frame, scheme="holdout", fold=None):
    """Return {'train','val','test'} -> row indices, after checking for group leakage."""
    if scheme == "holdout":
        subset, group = frame["holdout"].values, "mmsi_group"
    elif scheme in ("cv_mmsi", "cv_scene"):
        if fold is None:
            raise ValueError(f"{scheme} needs a fold number (0 to {cfg.n_folds - 1})")
        col = "fold_mmsi" if scheme == "cv_mmsi" else "fold_scene"
        subset = fold_split(frame, col, fold, cfg.n_folds)
        group = "mmsi_group" if scheme == "cv_mmsi" else "scene_group"
    else:
        raise ValueError(f"scheme must be one of {SCHEMES}")
    idx = {name: np.where(subset == name)[0] for name in ("train", "val", "test")}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        shared = set(frame.loc[idx[a], group]) & set(frame.loc[idx[b], group])
        assert not shared, f"LEAKAGE: {len(shared)} {group} values appear in both {a} and {b}"
    return idx

def summarize_split(frame, idx):
    names = ("train", "val", "test")
    tab = pd.DataFrame({n: frame.loc[idx[n], "ship_type"].value_counts() for n in names}).reindex(cfg.class_names).fillna(0).astype(int)
    tab.loc["TOTAL"] = tab.sum()
    print(tab.to_string())
    shore = pd.DataFrame({n: frame.loc[idx[n], "shoreline"].value_counts(normalize=True) * 100 for n in names}).round(1)
    print("\nShoreline share (%):")
    print(shore.to_string())

idx = get_split(df, "holdout")
print("Hold-out split (no vessel is shared between subsets):\n")
summarize_split(df, idx)

**Step 02 — Dataset and loaders.** Convert cached samples to tensors, normalize them, augment training data, and create DataLoaders.

In [ ]:
# @title
from torch.utils.data import Dataset, DataLoader

class SARDataset(Dataset):
    def __init__(self, X, labels, indices, prep, train):
        self.X, self.labels = X, labels

        self.indices, self.prep, self.train = np.asarray(indices), prep, train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, j):
        i = int(self.indices[j])
        x = torch.from_numpy(self.X[i].astype(np.float32)).unsqueeze(0)      # (1, S, S)
        x = self.prep.normalize(x)
        if self.train and self.prep.aug_enabled:
            x = self.prep.augment(x)
        x = self.prep.crop_to_input(x, train=self.train)
        return x, int(self.labels[i]), i

def make_loaders(X, labels, idx, prep, batch_size, num_workers=0, seed=42):
    gen = torch.Generator()
    gen.manual_seed(seed)
    loaders = {}
    for name in ("train", "val", "test"):
        train = name == "train"
        ds = SARDataset(X, labels, idx[name], prep, train)
        if train:
            assert len(ds) >= batch_size, f"only {len(ds)} training images for batch size {batch_size}"
        loaders[name] = DataLoader(ds, batch_size=batch_size if train else batch_size * 2,
                                   shuffle=train, drop_last=train, num_workers=num_workers,
                                   pin_memory=(device.type == "cuda"),
                                   generator=gen if train else None,
                                   persistent_workers=num_workers > 0)
    return loaders

**Step 03 — Prepare one run.** Load the recipe cache, split indices, training statistics, class weights, and loaders.

In [ ]:
# @title
def prepare_data(scheme="holdout", fold=None, prep=None, batch_size=None, num_workers=0, verbose=True):
    prep = prep if prep is not None else Preprocessor.from_cfg(cfg)
    batch_size = batch_size or cfg.batch_size
    X = get_cache(prep)
    labels = df["label"].values
    idx = get_split(df, scheme, fold)
    mean, std = prep.fit_normalization(X, idx["train"])
    counts = np.bincount(labels[idx["train"]], minlength=cfg.num_classes)
    weights = len(idx["train"]) / (cfg.num_classes * counts)
    loaders = make_loaders(X, labels, idx, prep, batch_size, num_workers, cfg.seed)
    if verbose:
        print(f"[data] {scheme}{'' if fold is None else f' fold {fold}'} | recipe {prep.tag} | input {prep.input_size}px | "
              f"train/val/test = {len(idx['train'])}/{len(idx['val'])}/{len(idx['test'])}")
        print(f"[data] normalisation (train only): mean {mean:.4f}, std {std:.4f}")
        print("[data] class weights:", {c: round(float(w), 2) for c, w in zip(cfg.class_names, weights)})
    return dict(prep=prep, X=X, labels=labels, idx=idx, loaders=loaders,
                class_weights=weights, counts=counts, scheme=scheme, fold=fold)

data = prepare_data("holdout")

**Step 04 — Sanity checks.** Check batch shape/statistics and inspect the augmentations applied to training images.

In [ ]:
# @title
tl = data["loaders"]["train"]
xb, yb, ib = next(iter(tl))
print("Batch:", tuple(xb.shape), xb.dtype, "| labels", yb.dtype)
print(f"Batch mean {xb.mean():.3f}, std {xb.std():.3f} (expect near 0 and 1)")

seen = np.zeros(cfg.num_classes, dtype=int)
for _, y, _ in tl:
    seen += np.bincount(y.numpy(), minlength=cfg.num_classes)
print("\nLabels seen in one training epoch:", dict(zip(cfg.class_names, seen.tolist())), f"({len(tl)} batches)")

ds_eval = SARDataset(data["X"], data["labels"], data["idx"]["train"], data["prep"], train=False)
ds_aug  = SARDataset(data["X"], data["labels"], data["idx"]["train"], data["prep"], train=True)
mean, std = data["prep"].norm_mean, data["prep"].norm_std

fig, axes = plt.subplots(cfg.num_classes, 4, figsize=(11, 2.8 * cfg.num_classes))
for r, c in enumerate(cfg.class_names):
    j = int(np.where(df.loc[data["idx"]["train"], "ship_type"].values == c)[0][0])
    views = [("eval view", ds_eval[j][0])] + [(f"augmented {k + 1}", ds_aug[j][0]) for k in range(3)]
    for k, (title, x) in enumerate(views):
        axes[r, k].imshow(x[0].numpy() * std + mean, cmap="gray")
        axes[r, k].set_title(f"{c}: {title}", fontsize=9)
        axes[r, k].axis("off")
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_augmentations.png", dpi=130)
plt.show()

# BASELINES & TRAINING PREPARATION

## Section 10 : Models and training engine
Define the shared training logic, custom CNN, pretrained CNNs, and Swin-Tiny.

**Step 01 — Run configuration and metrics.** Store run hyperparameters and define the common evaluation metrics.

In [ ]:
# @title
import copy, math, tempfile
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
from typing import Optional
from sklearn.metrics import (f1_score, balanced_accuracy_score, accuracy_score,
                             confusion_matrix, classification_report)

@dataclass
class RunConfig:
    arch: str = "custom"              # custom | resnet18 | efficientnet_b0 | swin_tiny
    pretrained: bool = False
    input_size: int = 224
    batch_size: int = 16
    lr: float = 3e-4
    weight_decay: float = 1e-4
    max_epochs: int = 100
    patience: int = 12
    min_delta: float = 1e-4
    label_smoothing: float = 0.0
    dropout: float = 0.4
    scheduler: str = "plateau"        # plateau | cosine
    plateau_patience: int = 5
    plateau_factor: float = 0.5
    warmup_epochs: int = 0
    grad_clip: float = 1.0
    amp: bool = True                  # mixed precision (used only on GPU)
    backbone_lr: float = 2e-5         # swin only
    head_lr: float = 1e-3             # swin only
    head_only_epochs: int = 5         # swin only
    split_scheme: str = "holdout"
    fold: Optional[int] = None
    seed: int = 42
    resume: bool = True

ARCH_DEFAULTS = {
    "custom":          dict(lr=cfg.learning_rate, weight_decay=cfg.weight_decay, max_epochs=cfg.max_epochs,
                            patience=cfg.patience, label_smoothing=cfg.label_smoothing, dropout=0.4, scheduler="plateau"),
    "resnet18":        dict(lr=1e-4, weight_decay=1e-4, max_epochs=60, patience=10,
                            label_smoothing=0.0, dropout=0.3, scheduler="plateau"),
    "efficientnet_b0": dict(lr=1e-4, weight_decay=1e-4, max_epochs=60, patience=10,
                            label_smoothing=0.0, dropout=0.3, scheduler="plateau"),
    "swin_tiny":       dict(lr=cfg.head_lr, weight_decay=0.05, max_epochs=40, patience=10,
                            label_smoothing=0.1, dropout=0.2, scheduler="cosine", warmup_epochs=2,
                            backbone_lr=cfg.backbone_lr, head_lr=cfg.head_lr,
                            head_only_epochs=cfg.head_only_epochs),
}

def make_run_config(arch, **overrides):
    d = dict(input_size=cfg.swin_input_size, batch_size=cfg.batch_size, seed=cfg.seed,
             pretrained=(arch != "custom"))
    d.update(ARCH_DEFAULTS[arch])
    d.update(overrides)
    return RunConfig(arch=arch, **d)

def compute_metrics(y_true, y_pred):
    return {"acc": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "bal_acc": float(balanced_accuracy_score(y_true, y_pred))}

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(logits, y, T=1.0):
    p = softmax_np(logits / T)
    return float(-np.log(p[np.arange(len(y)), y] + 1e-12).mean())

def fit_temperature(logits, y):
    grid = np.exp(np.linspace(np.log(0.3), np.log(10.0), 200))
    return float(grid[int(np.argmin([nll(logits, y, t) for t in grid]))])

def choose_threshold(probs, y, flag_fraction=0.2):
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    thr = float(np.quantile(conf, flag_fraction))
    keep = conf >= thr
    info = {"threshold": thr, "flagged_fraction": float((~keep).mean()),
            "acc_unflagged": float((pred[keep] == y[keep]).mean()),
            "acc_flagged": float((pred[~keep] == y[~keep]).mean()) if (~keep).any() else float("nan")}
    return thr, info

print("Helpers ready. Architectures:", list(ARCH_DEFAULTS))

**Step 02 — BaseModel.** Implement training, validation, early stopping, checkpoints, calibration, thresholds, and model-card saving.

In [ ]:
# @title
from abc import ABC, abstractmethod
import time, datetime

class BaseModel(ABC):
    """Class-diagram BaseModel: training engine shared by CNNModel and SwinTransformer."""

    def __init__(self, rc, class_names, device, run_dir):
        self.rc = rc
        self.model_name = rc.arch
        self.class_names = list(class_names)
        self.num_classes = len(self.class_names)
        self.device = device
        self.run_dir = Path(run_dir)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.class_weights = None
        self.temperature = 1.0
        self.threshold = 0.5
        self.history = []
        self.best_f1, self.best_epoch = -1.0, 0
        self.net = self.build_network().to(self.device)

    # ----- provided by CNNModel / SwinTransformer -----
    @abstractmethod
    def build_network(self):
        ...

    @abstractmethod
    def param_groups(self, stage="main"):
        ...

    @property
    def n_params(self):
        return sum(p.numel() for p in self.net.parameters())

    @property
    def n_trainable(self):
        return sum(p.numel() for p in self.net.parameters() if p.requires_grad)

    # ----- one epoch (train if opt is given, else validate) -----
    def _run_epoch(self, loader, criterion, opt=None, scaler=None, use_amp=False):
        train = opt is not None
        self.net.train(train)
        total, n, ys, ps = 0.0, 0, [], []
        with torch.set_grad_enabled(train):
            for x, y, _ in loader:
                x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
                with torch.autocast(device_type=self.device.type, enabled=use_amp):
                    logits = self.net(x)
                loss = criterion(logits.float(), y)
                if train:
                    opt.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    if self.rc.grad_clip:
                        scaler.unscale_(opt)
                        nn.utils.clip_grad_norm_(self.net.parameters(), self.rc.grad_clip)
                    scaler.step(opt)
                    scaler.update()
                total += loss.item() * len(y)
                n += len(y)
                ys.append(y.cpu().numpy())
                ps.append(logits.argmax(1).cpu().numpy())
        m = compute_metrics(np.concatenate(ys), np.concatenate(ps))
        m["loss"] = total / n
        return m

    # ----- one training stage -----
    def train(self, train_loader, val_loader, class_weights=None, epochs=None, stage="main",
              groups=None, scheduler=None, warmup_epochs=None, patience=None, resume=None):
        rc = self.rc
        epochs = rc.max_epochs if epochs is None else epochs
        patience = rc.patience if patience is None else patience
        sched_name = scheduler or rc.scheduler
        warmup = rc.warmup_epochs if warmup_epochs is None else warmup_epochs
        resume = rc.resume if resume is None else resume

        if class_weights is not None:
            self.class_weights = torch.as_tensor(class_weights, dtype=torch.float32)
        w = None if self.class_weights is None else self.class_weights.to(self.device)
        criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=rc.label_smoothing)

        opt = torch.optim.AdamW(groups or self.param_groups(stage), lr=rc.lr, weight_decay=rc.weight_decay)
        if sched_name == "cosine":
            def lr_lambda(e):
                if e < warmup:
                    return (e + 1) / max(1, warmup)
                p = (e - warmup) / max(1, epochs - warmup)
                return max(0.01, 0.5 * (1 + math.cos(math.pi * min(1.0, p))))
            sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        else:
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="max", factor=rc.plateau_factor, patience=rc.plateau_patience)

        use_amp = bool(rc.amp) and self.device.type == "cuda"
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        ck_path = self.run_dir / f"last_{stage}.pt"
        start, bad = 0, 0
        if resume and ck_path.exists():
            ck = torch.load(ck_path, map_location=self.device, weights_only=False)
            self.net.load_state_dict(ck["net"])
            opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"])
            scaler.load_state_dict(ck["scaler"])
            start, bad = ck["epoch"], ck["bad"]
            self.best_f1, self.best_epoch, self.history = ck["best_f1"], ck["best_epoch"], ck["history"]
            print(f"[resume] stage '{stage}' continues from epoch {start}")
            if ck.get("done"):
                print(f"[resume] stage '{stage}' was already finished")
                return pd.DataFrame(self.history)

        def save_ck(epoch, done=False):
            torch.save({"net": self.net.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": epoch, "bad": bad, "best_f1": self.best_f1,
                        "best_epoch": self.best_epoch, "history": self.history, "done": done}, ck_path)

        done_epochs = start
        for epoch in range(start, epochs):
            t0 = time.time()
            tr = self._run_epoch(train_loader, criterion, opt, scaler, use_amp)
            va = self._run_epoch(val_loader, criterion, None, None, use_amp)
            lr_now = opt.param_groups[0]["lr"]
            if sched_name == "cosine":
                sched.step()
            else:
                sched.step(va["macro_f1"])
            g_epoch = len(self.history) + 1
            row = {"epoch": g_epoch, "stage": stage, "lr": lr_now,
                   **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()},
                   "secs": time.time() - t0}
            self.history.append(row)
            improved = va["macro_f1"] > self.best_f1 + rc.min_delta
            if improved:
                self.best_f1, self.best_epoch, bad = va["macro_f1"], g_epoch, 0
                torch.save({"net": self.net.state_dict(), "epoch": g_epoch, "val_macro_f1": self.best_f1},
                           self.run_dir / "best.pt")
            else:
                bad += 1
            print(f"[{stage}] ep {g_epoch:3d} | lr {lr_now:.1e} | train loss {tr['loss']:.3f} f1 {tr['macro_f1']:.3f} | "
                  f"val loss {va['loss']:.3f} acc {va['acc']:.3f} f1 {va['macro_f1']:.3f}{' *' if improved else ''} | {row['secs']:.0f}s")
            done_epochs = epoch + 1
            save_ck(done_epochs)
            pd.DataFrame(self.history).to_csv(self.run_dir / "history.csv", index=False)
            if bad >= patience:
                print(f"[{stage}] early stop: no validation macro-F1 gain for {patience} epochs")
                break

        self.load_best()
        save_ck(done_epochs, done=True)
        print(f"[{stage}] best epoch {self.best_epoch} | val macro-F1 {self.best_f1:.3f}")
        return pd.DataFrame(self.history)

    def load_best(self):
        p = self.run_dir / "best.pt"
        if p.exists():
            self.net.load_state_dict(torch.load(p, map_location=self.device, weights_only=False)["net"])

    # ----- prediction, calibration, evaluation -----
    @torch.no_grad()
    def predict_logits(self, loader):
        self.net.eval()
        L, Y, I = [], [], []
        for x, y, i in loader:
            L.append(self.net(x.to(self.device)).float().cpu().numpy())
            Y.append(np.asarray(y))
            I.append(np.asarray(i))
        return np.concatenate(L), np.concatenate(Y), np.concatenate(I)

    @torch.no_grad()
    def predict_proba(self, x):
        """x: preprocessed tensor (1,H,W) or (B,1,H,W). Returns calibrated probabilities (B, K)."""
        self.net.eval()
        x = torch.as_tensor(x, dtype=torch.float32)
        if x.ndim == 3:
            x = x.unsqueeze(0)
        logits = self.net(x.to(self.device)).float().cpu().numpy()
        return softmax_np(logits / self.temperature)

    def predict(self, x):
        return [self.class_names[i] for i in self.predict_proba(x).argmax(axis=1)]

    def calibrate(self, val_loader):
        logits, y, _ = self.predict_logits(val_loader)
        self.temperature = fit_temperature(logits, y)
        print(f"[calibrate] temperature {self.temperature:.2f} | val log-loss "
              f"{nll(logits, y, 1.0):.3f} -> {nll(logits, y, self.temperature):.3f}")
        return self.temperature

    def choose_threshold(self, val_loader, flag_fraction=0.2):
        logits, y, _ = self.predict_logits(val_loader)
        self.threshold, info = choose_threshold(softmax_np(logits / self.temperature), y, flag_fraction)
        print(f"[threshold] {self.threshold:.2f} | flags {info['flagged_fraction']*100:.0f}% of val | "
              f"acc unflagged {info['acc_unflagged']:.3f} vs flagged {info['acc_flagged']:.3f}")
        return self.threshold, info

    def evaluate(self, loader):
        logits, y, idx = self.predict_logits(loader)
        probs = softmax_np(logits / self.temperature)
        m = compute_metrics(y, probs.argmax(axis=1))
        m["nll"] = nll(logits, y, self.temperature)
        return m, probs, y, idx

    # ----- saving -----
    def save_model(self):
        p = self.run_dir / "weights.pt"
        torch.save(self.net.state_dict(), p)
        return p

    def load_model(self, path=None):
        p = Path(path) if path else self.run_dir / "weights.pt"
        self.net.load_state_dict(torch.load(p, map_location=self.device))
        return self

    def save_model_card(self, prep, extra=None):
        card = {"model_name": self.model_name, "pretrained": self.rc.pretrained,
                "class_names": self.class_names, "num_classes": self.num_classes,
                "weights_file": "weights.pt", "preprocessing": prep.state(),
                "temperature": self.temperature, "threshold": self.threshold,
                "best_epoch": self.best_epoch, "best_val_macro_f1": self.best_f1,
                "run_config": asdict(self.rc), "n_params": self.n_params,
                "created": datetime.datetime.now().isoformat(timespec="seconds"),
                "torch": torch.__version__, "timm": timm.__version__, **(extra or {})}
        (self.run_dir / "model_card.json").write_text(json.dumps(card, indent=2))
        return card

**Step 03 — CNNModel.** Define the custom CNN and the ResNet-18/EfficientNet-B0 options for single-channel SAR input.

In [ ]:
# @title
class CustomCNN(nn.Module):
    def __init__(self, num_classes, in_chans=1, widths=(16, 32, 64, 128, 256), dropout=0.4):
        super().__init__()
        blocks, c_in = [], in_chans
        for i, w in enumerate(widths):
            blocks.append(nn.Sequential(
                nn.Conv2d(c_in, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(),
                nn.Conv2d(w, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU()))
            if i < len(widths) - 1:
                blocks.append(nn.MaxPool2d(2))
            c_in = w
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(widths[-1], num_classes)

    def forward(self, x):
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(self.drop(x))


class CNNModel(BaseModel):
    """Class-diagram CNNModel: architecture, pretrained, numLayers, dropoutRate."""

    def __init__(self, rc, class_names, device, run_dir):
        self.architecture = rc.arch
        self.pretrained = rc.pretrained
        self.dropout_rate = rc.dropout
        super().__init__(rc, class_names, device, run_dir)

    def build_architecture(self):
        if self.architecture == "custom":
            return CustomCNN(self.num_classes, in_chans=1, dropout=self.dropout_rate)
        if self.architecture in ("resnet18", "efficientnet_b0"):
            return timm.create_model(self.architecture, pretrained=self.pretrained,
                                     num_classes=self.num_classes, in_chans=1, drop_rate=self.dropout_rate)
        raise ValueError(f"Unknown CNN architecture: {self.architecture}")

    def build_network(self):
        return self.build_architecture()

    def param_groups(self, stage="main"):
        return [{"params": [p for p in self.net.parameters() if p.requires_grad], "lr": self.rc.lr}]

    @property
    def target_layer(self):
        if self.architecture == "custom":
            return self.net.features[-1]
        if self.architecture == "resnet18":
            return self.net.layer4[-1]
        return self.net.conv_head

**Step 04 — SwinTransformer.** Adapt pretrained Swin-Tiny to one input channel and a four-class output head.

In [ ]:
# @title
class SwinTransformer(BaseModel):
    """Class-diagram SwinTransformer: pretrainedWeights, windowSize, patchSize."""

    TIMM_NAME = "swin_tiny_patch4_window7_224"

    def __init__(self, rc, class_names, device, run_dir):
        if rc.input_size != 224:
            raise ValueError("Swin-Tiny expects 224 x 224 input (window 7 x patch 4).")
        self.pretrained_weights = f"timm/{self.TIMM_NAME} (ImageNet)" if rc.pretrained else "none (random init)"
        self.window_size, self.patch_size = 7, 4
        self.backbone_lr, self.head_lr = rc.backbone_lr, rc.head_lr
        super().__init__(rc, class_names, device, run_dir)

    def load_pretrained_weights(self):
        return timm.create_model(self.TIMM_NAME, pretrained=self.rc.pretrained, in_chans=1,
                                 drop_rate=self.rc.dropout, drop_path_rate=0.1)

    def replace_head(self, net=None):
        net = self.net if net is None else net
        net.reset_classifier(num_classes=self.num_classes)
        return net

    def build_network(self):
        return self.replace_head(self.load_pretrained_weights())

    def _head_params(self):
        return list(self.net.get_classifier().parameters())

    def _backbone_params(self):
        head_ids = {id(p) for p in self._head_params()}
        return [p for p in self.net.parameters() if id(p) not in head_ids]

    def freeze_backbone(self):
        for p in self._backbone_params():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.net.parameters():
            p.requires_grad = True

    def param_groups(self, stage="full"):
        if stage == "head":
            return [{"params": self._head_params(), "lr": self.head_lr}]
        return [{"params": self._backbone_params(), "lr": self.backbone_lr},
                {"params": self._head_params(), "lr": self.head_lr}]

    def fine_tune(self, train_loader, val_loader, class_weights=None):
        rc = self.rc
        if rc.head_only_epochs > 0:
            self.freeze_backbone()
            print(f"Stage 1: head only | trainable {self.n_trainable / 1e3:.1f}K of {self.n_params / 1e6:.1f}M parameters")
            self.train(train_loader, val_loader, class_weights, epochs=rc.head_only_epochs, stage="head",
                       groups=self.param_groups("head"), scheduler="cosine", warmup_epochs=0,
                       patience=rc.head_only_epochs + 1)
        self.unfreeze_backbone()
        print(f"Stage 2: full fine-tuning | trainable {self.n_trainable / 1e6:.1f}M parameters "
              f"(backbone lr {self.backbone_lr:g}, head lr {self.head_lr:g})")
        self.train(train_loader, val_loader, class_weights if rc.head_only_epochs == 0 else None,
                   epochs=rc.max_epochs, stage="full", groups=self.param_groups("full"),
                   scheduler=rc.scheduler, warmup_epochs=rc.warmup_epochs, patience=rc.patience)
        return pd.DataFrame(self.history)

**Step 05 — Model factory and reload.** Build architectures consistently and verify that saved weights can be reloaded.

In [ ]:
# @title
def make_model(rc, class_names, device, run_dir):
    if rc.arch == "swin_tiny":
        return SwinTransformer(rc, class_names, device, run_dir)
    return CNNModel(rc, class_names, device, run_dir)

def load_run(run_dir, dev=None):
    """Rebuild a trained model from its folder (weights.pt + model_card.json)."""
    run_dir, dev = Path(run_dir), dev or device
    card = json.loads((run_dir / "model_card.json").read_text())
    rc = RunConfig(**{**card["run_config"], "pretrained": False})   # weights come from weights.pt
    model = make_model(rc, card["class_names"], dev, run_dir)
    model.load_model(run_dir / card["weights_file"])
    model.temperature, model.threshold = card["temperature"], card["threshold"]
    model.net.eval()
    return model, card

# ---- smoke test ----
set_seed(cfg.seed)
x = torch.randn(2, 1, 224, 224)
y = torch.tensor([0, 1])
print(f"{'architecture':<16}{'params':>10}   output   initial loss")
for arch in ("custom", "resnet18", "efficientnet_b0", "swin_tiny"):
    rc = make_run_config(arch, pretrained=False)
    m = make_model(rc, cfg.class_names, device, tempfile.mkdtemp())
    m.net.eval()
    with torch.no_grad():
        out = m.net(x.to(device))
    loss = F.cross_entropy(out.float().cpu(), y).item()
    print(f"{arch:<16}{m.n_params / 1e6:>9.2f}M   {tuple(out.shape)}   {loss:.3f}")
    del m
print(f"(an untrained {cfg.num_classes}-class network should give about ln({cfg.num_classes}) = {math.log(cfg.num_classes):.3f})")

# ---- round trip: save, reload, compare ----
rc = make_run_config("custom", pretrained=False)
rt_dir = Path(tempfile.mkdtemp())
m = make_model(rc, cfg.class_names, device, rt_dir)
m.temperature = 1.7
m.save_model()
m.save_model_card(Preprocessor.from_cfg(cfg))
m2, card = load_run(rt_dir)
m.net.eval()
with torch.no_grad():
    same = torch.allclose(m.net(x.to(device)), m2.net(x.to(device)), atol=1e-6)
print("\nSave/load round trip identical:", same, "| temperature restored:", m2.temperature)
del m, m2

## CNN training
Train the custom CNN and ResNet-18 on the development split. Use a GPU for full experiments.

**Step 01 — Training sanity checks.** Verify initial loss, output shape, and that a small batch can be overfit.

In [ ]:
# @title
def overfit_check(arch="custom", n_img=16, steps=40, lr=1e-3):
    """A network that cannot memorise a handful of images has a bug."""
    d = prepare_data("holdout", verbose=False)
    ds = SARDataset(d["X"], d["labels"], d["idx"]["train"][:n_img], d["prep"], train=False)
    x = torch.stack([ds[j][0] for j in range(n_img)]).to(device)
    y = torch.tensor([ds[j][1] for j in range(n_img)]).to(device)
    print("Classes in the tiny set:", dict(zip(cfg.class_names,
          torch.bincount(y.cpu(), minlength=cfg.num_classes).tolist())))

    set_seed(cfg.seed)
    m = make_model(make_run_config(arch, pretrained=False), cfg.class_names, device, tempfile.mkdtemp())
    m.net.eval()
    with torch.no_grad():
        init = F.cross_entropy(m.net(x).float(), y).item()
    print(f"Initial loss {init:.3f} (expect about ln({cfg.num_classes}) = {math.log(cfg.num_classes):.3f})")

    opt = torch.optim.Adam(m.net.parameters(), lr=lr)
    m.net.train()
    for s in range(steps):
        logits = m.net(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 10 == 0 or s == steps - 1:
            acc = (logits.argmax(1) == y).float().mean().item()
            print(f"step {s:3d} | loss {loss.item():.3f} | train acc {acc:.2f}")
    print("OK: the model can learn." if acc >= 0.9 else
          "WARNING: it did not memorise the tiny set. Check labels, normalisation and learning rate before training.")
    return acc

_ = overfit_check()

**Step 02 — Experiment runner.** Share data preparation, training, validation, calibration, model saving, and run logging.

In [ ]:
# @title
RUNS_CSV = cfg.subdir("results") / "runs.csv"

def log_run(row):
    new = pd.DataFrame([row])
    if RUNS_CSV.exists():
        old = pd.read_csv(RUNS_CSV)
        new = pd.concat([old[old["run_name"] != row["run_name"]], new], ignore_index=True)
    new.to_csv(RUNS_CSV, index=False)
    return new

def run_experiment(arch, scheme="holdout", fold=None, pretrained=None, rc_overrides=None,
                   prep_overrides=None, tag="", save_test_preds=False, num_workers=0):
    pretrained = (arch != "custom") if pretrained is None else pretrained
    kw = dict(pretrained=pretrained, split_scheme=scheme, fold=fold)
    kw.update(rc_overrides or {})
    rc = make_run_config(arch, **kw)
    prep = Preprocessor.from_cfg(cfg, input_size=rc.input_size, **(prep_overrides or {}))
    name = (f"{arch}{'-pre' if rc.pretrained else ''}_{scheme}{'' if fold is None else f'-f{fold}'}"
            f"_{prep.tag}{tag}_s{rc.seed}")
    run_dir = cfg.subdir("models") / name

    set_seed(rc.seed)
    t0 = time.time()
    d = prepare_data(scheme, fold, prep, rc.batch_size, num_workers)
    model = make_model(rc, cfg.class_names, device, run_dir)
    print(f"=== {name} | {model.n_params / 1e6:.2f}M parameters | device {device} ===")
    tr_loader, va_loader = d["loaders"]["train"], d["loaders"]["val"]
    if arch == "swin_tiny":
        model.fine_tune(tr_loader, va_loader, d["class_weights"])
    else:
        model.train(tr_loader, va_loader, d["class_weights"])

    model.calibrate(va_loader)
    thr, info = model.choose_threshold(va_loader)
    val_m, val_probs, val_y, val_idx = model.evaluate(va_loader)
    model.save_model()
    card = model.save_model_card(prep, extra={"split_scheme": scheme, "fold": fold, "run_name": name,
                                              "threshold_info": info})
    np.savez(run_dir / "val_preds.npz", probs=val_probs, labels=val_y, index=val_idx)
    if save_test_preds:
        _, tp, ty, ti = model.evaluate(d["loaders"]["test"])
        np.savez(run_dir / "test_preds.npz", probs=tp, labels=ty, index=ti)
        print("[saved] test predictions (not displayed: they are for the final evaluation in Section 10)")

    hist = pd.DataFrame(model.history)
    row = dict(run_name=name, arch=arch, pretrained=rc.pretrained, scheme=scheme, fold=fold,
               recipe=prep.tag, seed=rc.seed, best_epoch=model.best_epoch, epochs_run=len(hist),
               val_acc=val_m["acc"], val_macro_f1=val_m["macro_f1"], val_bal_acc=val_m["bal_acc"],
               temperature=model.temperature, threshold=model.threshold,
               params_M=model.n_params / 1e6, minutes=(time.time() - t0) / 60,
               finished=time.strftime("%Y-%m-%d %H:%M"))
    log_run(row)
    print(f"=== done in {row['minutes']:.1f} min | val acc {val_m['acc']:.3f} | "
          f"val macro-F1 {val_m['macro_f1']:.3f} | saved to {run_dir.name} ===")
    return dict(name=name, model=model, data=d, prep=prep, rc=rc, history=hist, val=val_m,
                val_probs=val_probs, val_y=val_y, val_idx=val_idx, card=card, row=row)

# ---------- plotting helpers ----------
def plot_history(hist, title, path=None):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for a, (k, t) in zip(ax, [("loss", "Loss"), ("acc", "Accuracy"), ("macro_f1", "Macro-F1")]):
        a.plot(hist["epoch"], hist[f"train_{k}"], label="train")
        a.plot(hist["epoch"], hist[f"val_{k}"], label="validation")
        if hist["stage"].nunique() > 1:
            for e in hist.index[hist["stage"] != hist["stage"].shift()][1:]:
                a.axvline(hist.loc[e, "epoch"] - 0.5, color="gray", ls=":", lw=1)
        best = hist["val_macro_f1"].idxmax()
        a.axvline(hist.loc[best, "epoch"], color="green", ls="--", lw=1, alpha=0.6)
        a.set_title(t)
        a.set_xlabel("epoch")
        a.grid(alpha=0.3)
        a.legend()
    fig.suptitle(title)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=130)
    plt.show()

def plot_confusion(y, pred, title, path=None):
    K = cfg.num_classes
    cm = confusion_matrix(y, pred, labels=list(range(K)), normalize="true")
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K))
    ax.set_xticklabels(cfg.class_names, rotation=30)
    ax.set_yticks(range(K))
    ax.set_yticklabels(cfg.class_names)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=130)
    plt.show()

def show_run(r):
    fig_dir = cfg.subdir("figures")
    pred = r["val_probs"].argmax(1)
    print(f"{r['name']}\nvalidation: acc {r['val']['acc']:.3f} | macro-F1 {r['val']['macro_f1']:.3f} | "
          f"balanced acc {r['val']['bal_acc']:.3f} | temperature {r['model'].temperature:.2f} | "
          f"low-confidence threshold {r['model'].threshold:.2f}")
    plot_history(r["history"], r["name"], fig_dir / f"curves_{r['name']}.png")
    plot_confusion(r["val_y"], pred, f"Validation: {r['name']}", fig_dir / f"confusion_val_{r['name']}.png")
    print(classification_report(r["val_y"], pred, labels=list(range(cfg.num_classes)),
                                target_names=cfg.class_names, digits=3, zero_division=0))

print("Runner ready. Runs are logged to", RUNS_CSV)

**Step 03 — Train the custom CNN.** Fit the from-scratch CNN and select its checkpoint by validation macro-F1.

In [ ]:
# @title
res_cnn = run_experiment("custom")

**Step 04 — Inspect the custom CNN.** Review validation curves, confusion matrix, and per-class scores.

In [ ]:
# @title
show_run(res_cnn)

**Step 05 — Train ResNet-18.** Fine-tune the one-channel pretrained CNN with the same split and evaluation recipe.

In [ ]:
# @title
res_resnet = run_experiment("resnet18")
show_run(res_resnet)

In [ ]:
# @title
RUN_EXTRA_TUNING = True

if RUN_EXTRA_TUNING:
    tuning = {
        "custom_cos": ("custom", dict(scheduler="cosine", warmup_epochs=3, max_epochs=80, patience=80, lr=1e-3), {}),
        "custom_noaug": ("custom", dict(), dict(aug_enabled=False)),
        "resnet_e100": ("resnet18", dict(max_epochs=100, patience=15), {}),
        "resnet_cos": ("resnet18", dict(scheduler="cosine", warmup_epochs=2, max_epochs=60, patience=60, lr=3e-4), {}),
    }
    for tag, (arch, rc_overrides, prep_overrides) in tuning.items():
        run_experiment(arch, rc_overrides=rc_overrides, prep_overrides=prep_overrides, tag="_" + tag)
    print(pd.read_csv(RUNS_CSV)[["run_name", "val_acc", "val_macro_f1", "val_bal_acc", "best_epoch", "minutes"]].round(3).to_string(index=False))
else:
    print("Extra tuning skipped. Enable RUN_EXTRA_TUNING only for a documented experiment.")

**Step 06 — Compare CNN runs.** Review validation metrics against majority-class and AIS length/width baselines.

In [ ]:
# @title
runs = pd.read_csv(RUNS_CSV)
cols = ["run_name", "val_acc", "val_macro_f1", "val_bal_acc", "best_epoch", "temperature",
        "threshold", "params_M", "minutes"]
print("Logged runs:")
display(runs[cols].round(3))

# ---- reference baselines on the same hold-out validation set ----
from sklearn.tree import DecisionTreeClassifier
split = get_split(df, "holdout")
y_tr, y_va = df.loc[split["train"], "label"].values, df.loc[split["val"], "label"].values
ref = {}
majority = np.bincount(y_tr).argmax()
ref["Always the most common class"] = compute_metrics(y_va, np.full_like(y_va, majority))
if {"length", "width"} <= set(df.columns):
    feats = df[["length", "width"]].where(lambda t: t > 0)
    med = feats.loc[split["train"]].median()
    feats = feats.fillna(med).values
    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=cfg.seed)
    tree.fit(feats[split["train"]], y_tr)
    ref["AIS length + width only (no image)"] = compute_metrics(y_va, tree.predict(feats[split["val"]]))
print("\nReference baselines (same validation set):")
display(pd.DataFrame(ref).T.round(3))

**Step 07 — Optional experiments.** Run extra tuning, preprocessing ablations, or preliminary CV only when explicitly enabled.

In [ ]:
# @title
def run_preprocessing_ablation(arch="custom"):
    out = [run_experiment(arch, prep_overrides=dict(intensity_mode=mode))
           for mode in Preprocessor.MODES]
    out.append(run_experiment(arch, prep_overrides=dict(crop_mode="center_crop")))
    return out


def run_cv(arch, scheme="cv_mmsi", **kw):
    return [run_experiment(arch, scheme=scheme, fold=k, save_test_preds=True, **kw)
            for k in range(cfg.n_folds)]


RUN_ABLATION = False
RUN_CV_CNN = False

if RUN_ABLATION:
    ablation_results = run_preprocessing_ablation()
if RUN_CV_CNN:
    cv_cnn = run_cv("custom", tag="_development")

print("Ablation:", RUN_ABLATION, "| preliminary custom CV:", RUN_CV_CNN)
print("Final cross-validation is configured in Section 10.")

## Swin-Tiny fine-tuning
Compare an ImageNet-pretrained transformer with the CNNs using the same data and metrics.

**Step 01 — Inspect Swin-Tiny.** Check one-channel input, four output logits, parameter counts, and trainable stages.

In [ ]:
# @title
rc_probe = make_run_config("swin_tiny", pretrained=True)
probe = SwinTransformer(rc_probe, cfg.class_names, device, tempfile.mkdtemp())
print("Pretrained weights :", probe.pretrained_weights)
print("Parameters         :", f"{probe.n_params / 1e6:.2f}M")
print("Patch-embed conv   :", tuple(probe.net.patch_embed.proj.weight.shape), "(out, in, kH, kW)")

probe.freeze_backbone()
print("Stage 1 trainable  :", f"{probe.n_trainable / 1e3:.1f}K parameters (the head only)")
probe.unfreeze_backbone()
print("Stage 2 trainable  :", f"{probe.n_trainable / 1e6:.2f}M parameters (everything)")

probe.net.eval()
with torch.no_grad():
    out = probe.net(torch.randn(2, 1, 224, 224).to(device))
print("Output shape       :", tuple(out.shape), f"-> {cfg.num_classes} class scores per image")
del probe, out
if device.type == "cuda":
    torch.cuda.empty_cache()

**Step 02 — Fine-tune Swin-Tiny.** Train the classification head first, then unfreeze and fine-tune the backbone.

In [ ]:
# @title
res_swin = run_experiment("swin_tiny")

**Step 03 — Inspect Swin results.** Review validation curves and per-class metrics; do not treat validation as final test evidence.

In [ ]:
# @title
show_run(res_swin)

**Step 04 — Compare validation results.** Place the CNN and Swin validation results side by side for development only.

In [ ]:
# @title
named = {"Custom CNN": globals().get("res_cnn"),
         "ResNet-18 (pretrained)": globals().get("res_resnet"),
         "Swin-T (pretrained)": globals().get("res_swin")}
rows = []
for label, r in named.items():
    if r is None:
        continue
    pred = r["val_probs"].argmax(1)
    f1c = f1_score(r["val_y"], pred, average=None, labels=list(range(cfg.num_classes)), zero_division=0)
    rows.append({"model": label, "val acc": r["val"]["acc"], "val macro-F1": r["val"]["macro_f1"],
                 **{f"F1 {c}": f for c, f in zip(cfg.class_names, f1c)}})

if not rows:
    print("No runs in memory (the session was restarted). See results/runs.csv for the logged runs.")
else:
    cmp = pd.DataFrame(rows).set_index("model").round(3)
    display(cmp)
    ax = cmp[["val acc", "val macro-F1"]].plot(kind="bar", figsize=(7, 4), rot=15, ylim=(0, 1))
    ax.set_title("Validation comparison (development split, about 180 images)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_val_model_comparison.png", dpi=130)
    plt.show()

# EVALUATION
Use vessel-grouped out-of-fold predictions for the main comparison, then evaluate final holdout models once.

**Step 01 — Final runs (GPU).** Enable `RUN_FINAL_CV` and `RUN_FINAL_HOLDOUT` in the next cell when the final experiments are ready. Only `_final` runs from the current seed enter final comparison; holdout runs use the same tag for deployment.

In [ ]:
# @title
RUN_FINAL_CV = False
FINAL_CV_MODELS = ["custom", "resnet18", "swin_tiny"]
FINAL_CV_SCHEMES = ["cv_mmsi"]

if RUN_FINAL_CV:
    print("******************* RUNNING FINAL CV *******************")
    if "run_cv" not in globals():
        raise RuntimeError("Run Sections 05 to 08 in this session first.")
    for scheme in FINAL_CV_SCHEMES:
        for arch in FINAL_CV_MODELS:
            print(f"\n######## {arch} | {scheme} ########")
            run_cv(arch, scheme=scheme, tag="_final")

RUN_FINAL_HOLDOUT = False
if RUN_FINAL_HOLDOUT:
  print("******************* RUNNING FINAL HOLDOUT *******************")
  for arch in FINAL_CV_MODELS:
      print(f"\n######## {arch} | final holdout ########")
      run_experiment(arch, tag="_final")

RUN_EFFICIENTNET = False
if RUN_EFFICIENTNET:
  print("******************* RUNNING EFFICIENT NET *******************")
  run_cv("efficientnet_b0", scheme="cv_mmsi", tag="_final",
          prep_overrides=dict(aug_enabled=False))
  run_experiment("efficientnet_b0", tag="_final",
                   prep_overrides=dict(aug_enabled=False))

print("Final CV requested:", RUN_FINAL_CV,
      "| final holdout requested:", RUN_FINAL_HOLDOUT,
      "| optional EfficientNet:", RUN_EFFICIENTNET)
print("Enable the required flags once, after confirming the GPU/runtime and saved-run state.")

**Step 02 — Load cross-validation predictions.** Validate fold coverage and assemble one out-of-fold prediction per image.

In [ ]:
# @title
import re, itertools
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

RUNS_CSV    = cfg.subdir("results") / "runs.csv"
MODELS_DIR  = cfg.subdir("models")
RAW_NPY     = CACHE_DIR / "raw_uint16.npy"
MAIN_SCHEME = "cv_mmsi"
DEFAULT_RECIPE = "log_resize_256"
K = cfg.num_classes
ARCH_LABEL = {"custom": "Custom CNN", "resnet18": "ResNet-18",
              "efficientnet_b0": "EfficientNet-B0", "swin_tiny": "Swin-T"}

def fold_col(scheme):
    return {"cv_mmsi": "fold_mmsi", "cv_scene": "fold_scene"}[scheme]

def fold_indices(scheme, k):
    """Same rule as Section 06: fold k is test, fold k+1 is validation, the rest is training."""
    f = df[fold_col(scheme)].values
    nxt = (k + 1) % cfg.n_folds
    return {"train": np.where((f != k) & (f != nxt))[0], "val": np.where(f == nxt)[0],
            "test": np.where(f == k)[0]}

def model_label(arch, pretrained, recipe, tag, seed):
    name = ARCH_LABEL.get(arch, arch) + (" (pretrained)" if pretrained else "")
    extra = []
    if recipe != DEFAULT_RECIPE:
        extra.append(recipe)
    if tag:
        extra.append(tag.lstrip("_-"))
    if seed != cfg.seed:
        extra.append(f"seed {seed}")
    return name + (f" [{', '.join(extra)}]" if extra else "")

def load_cv_stores():
    if not RUNS_CSV.exists():
        return {}
    runs = pd.read_csv(RUNS_CSV)
    runs = runs[runs["fold"].notna() & runs["scheme"].isin(["cv_mmsi", "cv_scene"])]
    stores = {}
    for _, r in runs.iterrows():
        p = MODELS_DIR / r["run_name"] / "test_preds.npz"
        if not p.exists():
            continue
        fold, pre, seed = int(r["fold"]), bool(r["pretrained"]), int(r["seed"])
        prefix = f"{r['arch']}{'-pre' if pre else ''}_{r['scheme']}-f{fold}_{r['recipe']}"
        tag = r["run_name"][len(prefix):]
        if tag.endswith(f"_s{seed}"):
            tag = tag[:-len(f"_s{seed}")]
        key = (r["arch"], pre, r["scheme"], r["recipe"], tag, seed)
        z = np.load(p)
        st = stores.setdefault(key, dict(kind="model", arch=r["arch"], pretrained=pre, scheme=r["scheme"],
                                         recipe=r["recipe"], tag=tag, seed=seed, folds={},
                                         label=model_label(r["arch"], pre, r["recipe"], tag, seed)))
        st["folds"][fold] = dict(probs=z["probs"], labels=z["labels"], index=z["index"],
                                 threshold=float(r["threshold"]), run_name=r["run_name"])
    return stores

def assemble(st):
    """Combine the fold predictions into one out-of-fold matrix and validate it."""
    labels = df["label"].values
    probs = np.full((N, K), np.nan)
    fold_of, thr_of = np.full(N, -1), np.full(N, np.nan)
    for k, f in st["folds"].items():
        idx = f["index"]
        assert (labels[idx] == f["labels"]).all(), f"{st['label']} fold {k}: labels differ from splits.csv (stale run?)"
        assert (df[fold_col(st["scheme"])].values[idx] == k).all(), f"{st['label']} fold {k}: images are not in that fold"
        probs[idx], fold_of[idx], thr_of[idx] = f["probs"], k, f["threshold"]
    st.update(probs=probs, fold_of=fold_of, thr_of=thr_of, covered=~np.isnan(probs[:, 0]))
    return st

stores = load_cv_stores()
final_suffix = f"_final_s{cfg.seed}"
stores = {key: st for key, st in stores.items()
          if st["scheme"] in ("cv_mmsi", "cv_scene")
          and st["tag"] == "_final" and st["seed"] == cfg.seed
          and all(run_name.endswith(final_suffix) for run_name in
                  [fold["run_name"] for fold in st["folds"].values()])}
for _st in stores.values():
    assemble(_st)
HAVE_CV = len(stores) > 0

def require_cv():
    if not HAVE_CV:
        raise RuntimeError("No cross-validated test predictions found. Run Step 01 with RUN_FINAL_CV = True "
                           "(or run_cv(...) from Section 08), then re-run this cell.")

if HAVE_CV:
    overview = pd.DataFrame([{"model": s["label"], "scheme": s["scheme"], "folds found": sorted(s["folds"]),
                              "images with a test prediction": f"{int(s['covered'].sum())} / {N}"}
                             for s in stores.values()])
    display(overview)
    incomplete = [s["label"] for s in stores.values() if len(s["folds"]) < cfg.n_folds]
    if incomplete:
        print("WARNING: incomplete folds for:", incomplete)
else:
    print("No cross-validated runs found yet. Use Step 01 (RUN_FINAL_CV = True) to create them.")

In [ ]:
# @title
KEEP_TAG = "_final"
stores = {k: s for k, s in stores.items() if s["tag"] == KEEP_TAG}
HAVE_CV = len(stores) > 0
print([s["label"] for s in stores.values()])

In [ ]:
# @title
MODELS_DIR = cfg.subdir("models")
IGNORE = {"split_scheme", "fold", "resume"}


def card_cfg(run_name):
    p = MODELS_DIR / run_name / "model_card.json"
    if not p.exists():
        return None
    c = json.loads(p.read_text())
    d = {k: v for k, v in c["run_config"].items() if k not in IGNORE}
    d["aug_enabled"] = c["preprocessing"].get("aug_enabled", "not recorded")
    d["recipe"] = f"{c['preprocessing']['intensity_mode']}_{c['preprocessing']['crop_mode']}_{c['preprocessing']['target_size']}"
    return d


pairs = {"Custom CNN": "custom", "ResNet-18": "resnet18-pre",
         "Swin-T": "swin_tiny-pre", "EfficientNet-B0": "efficientnet_b0-pre"}
for label, stem in pairs.items():
    a = card_cfg(f"{stem}_cv_mmsi-f0_log_resize_256_final_s{cfg.seed}")
    b = card_cfg(f"{stem}_holdout_log_resize_256_final_s{cfg.seed}")
    if a is None or b is None:
        print(f"{label}: skipped (final CV fold-0 or final holdout run not found)")
        continue
    diff = {k: (a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)}
    print(f"{label}: " + ("same settings in CV and holdout" if not diff else f"DIFFERENT (cv, holdout) -> {diff}"))

**Step 03 — Evaluator.** Calculate overall, per-class, subset, and calibration metrics from predictions.

In [ ]:
# @title
class Evaluator:
    """Class-diagram Evaluator: predictions, groundTruth, classNames, threshold."""

    def __init__(self, class_names, threshold=0.0):
        self.class_names = list(class_names)
        self.threshold = threshold
        self.probs = self.predictions = self.ground_truth = self.meta = None

    def set_data(self, probs, labels, meta=None):
        self.probs = np.asarray(probs, dtype=float)
        self.ground_truth = np.asarray(labels).astype(int)
        self.predictions = self.probs.argmax(axis=1)
        self.meta = None if meta is None else meta.reset_index(drop=True)
        return self

    def _pick(self, mask):
        if mask is None:
            return self.ground_truth, self.predictions
        mask = np.asarray(mask, dtype=bool)
        return self.ground_truth[mask], self.predictions[mask]

    def _macro(self, y, p):
        return precision_recall_fscore_support(y, p, labels=np.unique(y), average="macro", zero_division=0)

    def compute_accuracy(self, mask=None):
        y, p = self._pick(mask)
        return float(accuracy_score(y, p))

    def compute_precision(self, mask=None):
        return float(self._macro(*self._pick(mask))[0])

    def compute_recall(self, mask=None):
        return float(self._macro(*self._pick(mask))[1])

    def compute_f1_score(self, mask=None):
        return float(self._macro(*self._pick(mask))[2])

    def compute_all_metrics(self, mask=None):
        y, p = self._pick(mask)
        pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=list(range(len(self.class_names))),
                                                          zero_division=0)
        per_class = pd.DataFrame({"precision": pr, "recall": rc, "f1": f1, "support": sup},
                                 index=self.class_names)
        mp, mr, mf, _ = self._macro(y, p)
        summary = {"n": int(len(y)), "acc": float(accuracy_score(y, p)), "macro_precision": float(mp),
                   "macro_recall": float(mr), "macro_f1": float(mf), "bal_acc": float(balanced_accuracy_score(y, p))}
        return {"summary": summary, "per_class": per_class}

    def evaluate_by_subset(self, column):
        rows = []
        for value in sorted(self.meta[column].unique()):
            rows.append({column: value, **self.compute_all_metrics((self.meta[column] == value).values)["summary"]})
        return pd.DataFrame(rows).set_index(column)

    def generate_confusion_matrix(self, normalize=None, mask=None):
        y, p = self._pick(mask)
        cm = confusion_matrix(y, p, labels=list(range(len(self.class_names))), normalize=normalize)
        return np.nan_to_num(cm)

    def expected_calibration_error(self, bins=10):
        conf, correct = self.probs.max(axis=1), (self.predictions == self.ground_truth)
        edges = np.linspace(1.0 / len(self.class_names), 1.0, bins + 1)
        which = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
        ece = 0.0
        for b in range(bins):
            m = which == b
            if m.any():
                ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
        return float(ece)

    def flag_low_confidence(self):
        return self.probs.max(axis=1) < self.threshold

def reliability_points(probs, y, bins=10):
    conf, correct = probs.max(axis=1), (probs.argmax(axis=1) == y)
    edges = np.linspace(1.0 / probs.shape[1], 1.0, bins + 1)
    which = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
    pts = [(conf[which == b].mean(), correct[which == b].mean(), int((which == b).sum()))
           for b in range(bins) if (which == b).any()]
    return pts

def pooled_evaluator(st, meta_cols=None):
    cov = st["covered"]
    meta = df.loc[cov, meta_cols] if meta_cols else None
    return Evaluator(cfg.class_names, 0.0).set_data(st["probs"][cov], df["label"].values[cov], meta)

print("Evaluator ready.")

**Step 04 — Reference baselines.** Score majority-class and AIS length/width-only baselines on the same folds.

In [ ]:
# @title
from sklearn.tree import DecisionTreeClassifier

def make_baseline_stores(schemes):
    out = {}
    have_size = {"length", "width"} <= set(df.columns)
    feats = df[["length", "width"]].where(lambda t: t > 0) if have_size else None
    y_all = df["label"].values
    for scheme in schemes:
        for name, label in (("majority", "Always the most common class"),
                            ("size", "AIS length + width only (no image)")):
            if name == "size" and not have_size:
                continue
            st = dict(kind="baseline", label=label, arch=f"baseline-{name}", pretrained=False, scheme=scheme,
                      recipe="-", tag="", seed=0, folds={})
            for k in range(cfg.n_folds):
                sp = fold_indices(scheme, k)
                te, y_tr = sp["test"], y_all[sp["train"]]
                if name == "majority":
                    probs = np.zeros((len(te), K))
                    probs[:, np.bincount(y_tr, minlength=K).argmax()] = 1.0
                else:
                    med = feats.iloc[sp["train"]].median()
                    F = feats.fillna(med).values
                    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced",
                                                  random_state=cfg.seed).fit(F[sp["train"]], y_tr)
                    probs = np.zeros((len(te), K))
                    probs[:, tree.classes_] = tree.predict_proba(F[te])
                st["folds"][k] = dict(probs=probs, labels=y_all[te], index=te, threshold=0.0, run_name="baseline")
            out[(st["arch"], False, scheme, "-", "", 0)] = assemble(st)
    return out

if HAVE_CV:
    stores.update(make_baseline_stores(sorted({s["scheme"] for s in stores.values()})))
    print("Stores:", [(s["label"], s["scheme"]) for s in stores.values()])

**Step 05 — Headline results.** Report mean and standard deviation across the five vessel-grouped folds.

In [ ]:
# @title
METRICS = [("acc", "accuracy"), ("macro_f1", "macro-F1"), ("bal_acc", "balanced acc."),
           ("macro_precision", "macro precision"), ("macro_recall", "macro recall")]

def store_fold_table(st):
    ev = Evaluator(cfg.class_names)
    rows = []
    for k, f in sorted(st["folds"].items()):
        ev.set_data(f["probs"], f["labels"])
        rows.append({"model": st["label"], "kind": st["kind"], "scheme": st["scheme"], "fold": k,
                     **ev.compute_all_metrics()["summary"]})
    return pd.DataFrame(rows)

def summarize_folds(ft):
    cols = [m for m, _ in METRICS]
    g = ft.groupby(["scheme", "model", "kind"], sort=False)[cols]
    return g.mean(), g.std()

def format_mean_std(mean, std):
    out = pd.DataFrame(index=mean.index)
    for m, nice in METRICS:
        out[nice] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(mean[m], std[m])]
    return out

if HAVE_CV:
    FOLD_TABLE = pd.concat([store_fold_table(s) for s in stores.values()], ignore_index=True)
    FOLD_TABLE.to_csv(cfg.subdir("results") / "cv_fold_metrics.csv", index=False)
    HEAD_MEAN, HEAD_STD = summarize_folds(FOLD_TABLE)
    for scheme in sorted(FOLD_TABLE["scheme"].unique()):
        order = HEAD_MEAN.loc[scheme]["macro_f1"].sort_values(ascending=False).index
        shown = format_mean_std(HEAD_MEAN.loc[scheme], HEAD_STD.loc[scheme]).loc[order]
        print(f"\n=== {scheme}: mean ± std over {cfg.n_folds} folds (test folds only) ===")
        display(shown.reset_index().rename(columns={"model": "model", "kind": ""}))

**Step 06 — Per-class results.** Inspect pooled out-of-fold confusion matrices and class-level precision, recall, and F1.

In [ ]:
# @title
if HAVE_CV:
    main = [s for s in stores.values() if s["scheme"] == MAIN_SCHEME and s["covered"].all()]
    if not main:
        print(f"No model has complete {MAIN_SCHEME} coverage yet.")
    f1_tab = {}
    fig, axes = plt.subplots(1, max(1, len(main)), figsize=(4.3 * max(1, len(main)), 4.2))
    for a, st in zip(np.atleast_1d(axes), main):
        ev = pooled_evaluator(st)
        res = ev.compute_all_metrics()
        f1_tab[st["label"]] = res["per_class"]["f1"]
        cm = ev.generate_confusion_matrix(normalize="true")
        a.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        a.set_xticks(range(K))
        a.set_xticklabels(cfg.class_names, rotation=35, fontsize=8)
        a.set_yticks(range(K))
        a.set_yticklabels(cfg.class_names, fontsize=8)
        for i in range(K):
            for j in range(K):
                a.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                       color="white" if cm[i, j] > 0.5 else "black")
        a.set_title(st["label"], fontsize=9)
        a.set_xlabel("predicted")
        a.set_ylabel("true")
        print(f"\n--- {st['label']} ({MAIN_SCHEME}, pooled out-of-fold) ---")
        display(res["per_class"].round(3))
        raw_cm = ev.generate_confusion_matrix()
        off = [(raw_cm[i, j], cfg.class_names[i], cfg.class_names[j]) for i in range(K) for j in range(K) if i != j]
        print("Most common confusions (true -> predicted):",
              "; ".join(f"{t} -> {p} ({int(c)})" for c, t, p in sorted(off, reverse=True)[:3]))
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_cv_confusion_matrices.png", dpi=130)
    plt.show()

**Step 07 — Shoreline analysis.** Compare model performance for inshore and offshore samples.

In [ ]:
# @title
if HAVE_CV:
    rows = []
    for st in stores.values():
        if st["scheme"] != MAIN_SCHEME or not st["covered"].all():
            continue
        t = pooled_evaluator(st, ["shoreline"]).evaluate_by_subset("shoreline")
        for shore, r in t.iterrows():
            rows.append({"model": st["label"], "shoreline": shore, "n": int(r["n"]),
                         "acc": r["acc"], "macro_f1": r["macro_f1"]})
    SHORE = pd.DataFrame(rows)
    print("Images per class and shoreline:")
    display(pd.crosstab(df["ship_type"], df["shoreline"]).loc[cfg.class_names])
    display(SHORE.pivot(index="model", columns="shoreline", values=["acc", "macro_f1"]).round(3))
    ax = SHORE.pivot(index="model", columns="shoreline", values="macro_f1").plot(kind="bar", figsize=(8, 4), rot=20, ylim=(0, 1))
    ax.set_title(f"Macro-F1 by shoreline ({MAIN_SCHEME}, pooled out-of-fold)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_cv_shoreline.png", dpi=130)
    plt.show()

**Step 08 — Calibration.** Review reliability plots, ECE, and the low-confidence flag on held-out fold predictions.

In [ ]:
# @title
if HAVE_CV:
    models = [s for s in stores.values() if s["kind"] == "model" and s["scheme"] == MAIN_SCHEME and s["covered"].all()]
    rows = []
    fig, axes = plt.subplots(1, max(1, len(models)), figsize=(4.2 * max(1, len(models)), 4.0))
    for a, st in zip(np.atleast_1d(axes), models):
        ev = pooled_evaluator(st)
        ev.threshold = 0.0
        conf, pred, y = ev.probs.max(1), ev.predictions, ev.ground_truth
        flag = conf < st["thr_of"][st["covered"]]
        rows.append({"model": st["label"], "ECE": ev.expected_calibration_error(),
                     "mean confidence": conf.mean(), "accuracy": (pred == y).mean(),
                     "flagged share": flag.mean(),
                     "acc unflagged": (pred[~flag] == y[~flag]).mean() if (~flag).any() else np.nan,
                     "acc flagged": (pred[flag] == y[flag]).mean() if flag.any() else np.nan})
        pts = reliability_points(ev.probs, y)
        a.plot([1 / K, 1], [1 / K, 1], "k--", lw=1, label="perfect")
        a.plot([p[0] for p in pts], [p[1] for p in pts], "o-", label="model")
        a.set_title(st["label"], fontsize=9)
        a.set_xlabel("confidence")
        a.set_ylabel("accuracy")
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_cv_reliability.png", dpi=130)
    plt.show()
    CALIB = pd.DataFrame(rows).set_index("model")
    display(CALIB.round(3))

**Step 09 — Error analysis.** Inspect high-confidence mistakes and look for recurring image or metadata patterns.

In [ ]:
# @title
if HAVE_CV:
    cand = HEAD_MEAN.loc[MAIN_SCHEME].reset_index()
    cand = cand[cand["kind"] == "model"]
    best_label = cand.loc[cand["macro_f1"].idxmax(), "model"]
    best = next(s for s in stores.values() if s["label"] == best_label and s["scheme"] == MAIN_SCHEME)
    print("Best model by mean macro-F1:", best_label)

    y, probs = df["label"].values, best["probs"]
    cov = best["covered"]
    pred = np.where(cov, np.nan_to_num(probs).argmax(1), -1)
    wrong = cov & (pred != y)
    err = pd.DataFrame({"true": df["ship_type"], "predicted": [cfg.class_names[p] if p >= 0 else "" for p in pred],
                        "confidence": np.nan_to_num(probs).max(1), "shoreline": df["shoreline"],
                        "length_m": df["length"] if "length" in df.columns else np.nan,
                        "scene": df["scene_group"]})[wrong].sort_values("confidence", ascending=False)
    print(f"\nWrong predictions: {len(err)} of {int(cov.sum())} ({len(err) / cov.sum() * 100:.1f}%)")
    print("Most confident mistakes:")
    display(err.head(12).round(2))

    if RAW_NPY.exists():
        raw_view = np.load(RAW_NPY, mmap_mode="r")
        top = err.head(12)
        fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
        for a, (i, r) in zip(axes.ravel(), top.iterrows()):
            a.imshow(np.log1p(np.asarray(raw_view[i][128:384, 128:384], dtype=np.float32)), cmap="gray")
            a.set_title(f"{r['true']} -> {r['predicted']}\n{r['confidence']:.2f} | {r['shoreline']}", fontsize=8)
            a.axis("off")
        for a in axes.ravel()[len(top):]:
            a.axis("off")
        plt.tight_layout()
        plt.savefig(cfg.subdir("figures") / "fig_cv_confident_errors.png", dpi=130)
        plt.show()
    else:
        print("raw_uint16.npy not found: skipping the patch viewer.")

    if "length" in df.columns:
        valid = (df["length"] > 0).values & cov
        bins = pd.qcut(df.loc[valid, "length"], 4, duplicates="drop")
        acc = pd.DataFrame({"length bin (m)": bins.astype(str), best_label: (pred[valid] == y[valid])})
        base = next((s for s in stores.values() if s["arch"] == "baseline-size" and s["scheme"] == MAIN_SCHEME), None)
        if base is not None:
            acc["size-only baseline"] = base["probs"][valid].argmax(1) == y[valid]
        table = acc.groupby("length bin (m)", sort=False).mean().round(3)
        table.insert(0, "n", acc.groupby("length bin (m)", sort=False).size())
        print("\nAccuracy by ship-length quartile:")
        display(table)

**Step 10 — Locked holdout test.** Run the selected model's test evaluation only after Section 11 chooses the app model. The evaluation cell is placed after the model-selection step.

In [ ]:
# @title
print("Locked holdout evaluation is deferred until the  model-selection step.")

# MODEL COMPARISON
Compare networks and baselines, quantify uncertainty, and select the model used by the app.

**Step 01 — ModelComparator.** Summarize cross-validated metrics and highlight the best mean macro-F1.

In [ ]:
# @title
class ModelComparator:
    """Class-diagram ModelComparator."""

    def __init__(self, fold_table):
        self.fold_table = fold_table
        self.results = {}

    def load_results(self, model_name, scheme):
        t = self.fold_table
        rows = t[(t["model"] == model_name) & (t["scheme"] == scheme)]
        self.results[(model_name, scheme)] = rows.sort_values("fold").reset_index(drop=True)
        return self.results[(model_name, scheme)]

    def compare_metrics(self, model_names, scheme):
        rows = []
        for name in model_names:
            r = self.load_results(name, scheme)
            row = {"model": name, "folds": len(r)}
            for m, _ in METRICS:
                row[f"{m}_mean"], row[f"{m}_std"] = r[m].mean(), r[m].std()
            rows.append(row)
        return pd.DataFrame(rows).set_index("model")

    def generate_bar_charts(self, table, metrics=("acc", "macro_f1", "bal_acc"), title="", path=None):
        nice = dict(METRICS)
        fig, ax = plt.subplots(figsize=(max(7, 1.9 * len(table) + 3), 4.4))
        x, w = np.arange(len(table)), 0.8 / len(metrics)
        for j, m in enumerate(metrics):
            ax.bar(x + j * w, table[f"{m}_mean"], w, yerr=table[f"{m}_std"], capsize=3, label=nice[m])
        ax.set_xticks(x + w * (len(metrics) - 1) / 2)
        ax.set_xticklabels([n.replace(" (", "\n(") for n in table.index], fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        if path:
            plt.savefig(path, dpi=130)
        plt.show()

    def highlight_best_model(self, table, metric="macro_f1"):
        best = table[f"{metric}_mean"].idxmax()
        cut = table.loc[best, f"{metric}_mean"] - table.loc[best, f"{metric}_std"]
        tied = [n for n in table.index if n != best and table.loc[n, f"{metric}_mean"] >= cut]
        return best, tied

if HAVE_CV:
    comp = ModelComparator(FOLD_TABLE)
    names_main = list(dict.fromkeys(FOLD_TABLE.loc[FOLD_TABLE["scheme"] == MAIN_SCHEME, "model"]))
    kinds = FOLD_TABLE.drop_duplicates("model").set_index("model")["kind"]
    tab_all = comp.compare_metrics(names_main, MAIN_SCHEME).sort_values("macro_f1_mean", ascending=False)
    show = pd.DataFrame({nice: [f"{tab_all.loc[n, m + '_mean']:.3f} ± {tab_all.loc[n, m + '_std']:.3f}"
                                for n in tab_all.index] for m, nice in METRICS}, index=tab_all.index)
    show.insert(0, "kind", [kinds[n] for n in tab_all.index])
    print(f"=== {MAIN_SCHEME}: mean ± std over {cfg.n_folds} folds ===")
    display(show)
    comp.generate_bar_charts(tab_all, title=f"Cross-validated test results ({MAIN_SCHEME})",
                             path=cfg.subdir("figures") / "fig_model_comparison.png")
    nets = tab_all[[kinds[n] == "model" for n in tab_all.index]]
    best, tied = comp.highlight_best_model(nets)
    print("Best network by mean macro-F1:", best)
    print("Tied with it (within one std):", tied if tied else "none")
    base_size = [n for n in tab_all.index if kinds[n] == "baseline" and "length" in n]
    if base_size:
        gap = tab_all.loc[best, "macro_f1_mean"] - tab_all.loc[base_size[0], "macro_f1_mean"]
        print(f"Best network minus the size-only baseline (macro-F1): {gap:+.3f}")
else:
    print("No cross-validated runs: run Section 10 first.")

**Step 02 — Statistical comparison.** Review paired fold differences and the pooled significance tests with their limitations.

In [ ]:
# @title
from scipy.stats import binomtest

def paired_fold_table(scheme, metric="macro_f1"):
    wide = FOLD_TABLE[FOLD_TABLE["scheme"] == scheme].pivot(index="fold", columns="model", values=metric).dropna(axis=1)
    rows = []
    for a, b in itertools.combinations(wide.columns, 2):
        d = wide[a] - wide[b]
        rows.append({"A": a, "B": b, "mean(A-B)": d.mean(), "std": d.std(), "A wins": f"{int((d > 0).sum())}/{len(d)}"})
    return pd.DataFrame(rows).sort_values("mean(A-B)", ascending=False)

def macro_f1_batch(cms):
    """cms: (B, K, K) confusion matrices [true, pred] -> macro-F1 for each."""
    tp = np.einsum("bkk->bk", cms).astype(float)
    fp, fn = cms.sum(axis=1) - tp, cms.sum(axis=2) - tp
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean(axis=1)

def significance_table(scheme, n_boot=2000, seed=42):
    items = {s["label"]: s for s in stores.values() if s["scheme"] == scheme and s["covered"].all()}
    y = df["label"].values
    gcol = "mmsi_group" if scheme == "cv_mmsi" else "scene_group"
    codes, uniq = pd.factorize(df[gcol])
    G = len(uniq)
    W = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot)      # (B, G)
    cm_g, boot, full = {}, {}, {}
    for name, s in items.items():
        C = np.zeros((G, K, K), dtype=np.int64)
        np.add.at(C, (codes, y, s["probs"].argmax(1)), 1)
        cm_g[name] = C
        boot[name] = macro_f1_batch(np.tensordot(W, C, axes=(1, 0)))
        full[name] = float(macro_f1_batch(C.sum(0)[None])[0])
    rows = []
    for a, b in itertools.combinations(items, 2):
        ca, cb = items[a]["probs"].argmax(1) == y, items[b]["probs"].argmax(1) == y
        only_a, only_b = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p = binomtest(min(only_a, only_b), only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
        lo, hi = np.percentile(boot[a] - boot[b], [2.5, 97.5])
        rows.append({"A": a, "B": b, "macro-F1 A": full[a], "macro-F1 B": full[b], "diff A-B": full[a] - full[b],
                     "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "CI excludes 0": bool(lo > 0 or hi < 0),
                     "only A right": only_a, "only B right": only_b, "McNemar p": p})
    return pd.DataFrame(rows)

if HAVE_CV:
    print(f"=== Paired macro-F1 differences per fold ({MAIN_SCHEME}) ===")
    display(paired_fold_table(MAIN_SCHEME).round(3))
    sig = significance_table(MAIN_SCHEME)
    print(f"\n=== Pooled out-of-fold significance ({MAIN_SCHEME}, vessel bootstrap) ===")
    display(sig.round(3))
    sig.to_csv(cfg.subdir("results") / "cv_significance.csv", index=False)

**Step 03 — Per-class comparison.** Identify which classes benefit from pretraining or the transformer architecture.

In [ ]:
# @title
if HAVE_CV:
    rows = {}
    for s in stores.values():
        if s["scheme"] == MAIN_SCHEME and s["covered"].all():
            rows[s["label"]] = pooled_evaluator(s).compute_all_metrics()["per_class"]["f1"]
    PERCLASS_F1 = pd.DataFrame(rows).T
    display(PERCLASS_F1.round(3))
    ax = PERCLASS_F1.T.plot(kind="bar", figsize=(9, 4.2), rot=0, ylim=(0, 1))
    ax.set_title(f"Per-class F1 ({MAIN_SCHEME}, pooled out-of-fold)")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(cfg.subdir("figures") / "fig_per_class_f1.png", dpi=130)
    plt.show()

**Step 04 — Split robustness.** Compare vessel-grouped and scene-grouped CV when both are available.

In [ ]:
# @title
if HAVE_CV:
    schemes_present = sorted(FOLD_TABLE["scheme"].unique())
    if len(schemes_present) < 2:
        print("Only one split scheme has been run, so there is nothing to compare yet.")
    else:
        t = FOLD_TABLE.groupby(["model", "scheme"])["macro_f1"].agg(["mean", "std"]).unstack("scheme").dropna()
        out = pd.DataFrame({s: [f"{t[('mean', s)][m]:.3f} ± {t[('std', s)][m]:.3f}" for m in t.index]
                            for s in schemes_present}, index=t.index)
        if {"cv_mmsi", "cv_scene"} <= set(schemes_present):
            out["change (scene - vessel)"] = (t[("mean", "cv_scene")] - t[("mean", "cv_mmsi")]).round(3)
        display(out)

**Step 05 — Preprocessing ablation.** Compare validation results for the optional intensity/crop variants.

In [ ]:
# @title
runs_all = pd.read_csv(RUNS_CSV) if RUNS_CSV.exists() else pd.DataFrame()
if len(runs_all):
    h = runs_all[runs_all["scheme"] == "holdout"]
    multi = h.groupby(["arch", "pretrained"])["recipe"].nunique()
    keep = multi[multi > 1].index
    abl = h[[(a, p) in keep for a, p in zip(h["arch"], h["pretrained"])]]
    if len(abl):
        display(abl[["run_name", "arch", "recipe", "val_acc", "val_macro_f1", "val_bal_acc"]]
                .sort_values(["arch", "recipe"]).round(3).reset_index(drop=True))
    else:
        print("No ablation runs yet. Set RUN_ABLATION = True in Section 08 Step 07 (custom CNN, four short runs).")
else:
    print("results/runs.csv not found.")

**Step 06 — Optional ensemble.** Treat probability averaging as supplementary analysis, not the default deployment model.

In [ ]:
# @title
if HAVE_CV:
    nets_main = [s for s in stores.values() if s["kind"] == "model" and s["scheme"] == MAIN_SCHEME and len(s["folds"]) == cfg.n_folds]
    if len(nets_main) < 2:
        print("Need at least two complete networks for an ensemble.")
    else:
        ev, rows = Evaluator(cfg.class_names), []
        for k in range(cfg.n_folds):
            idx0 = nets_main[0]["folds"][k]["index"]
            assert all((s["folds"][k]["index"] == idx0).all() for s in nets_main), "test folds differ between models"
            ev.set_data(np.mean([s["folds"][k]["probs"] for s in nets_main], axis=0), nets_main[0]["folds"][k]["labels"])
            rows.append(ev.compute_all_metrics()["summary"])
        E = pd.DataFrame(rows)
        print("Ensemble of:", [s["label"] for s in nets_main])
        print(f"macro-F1 {E['macro_f1'].mean():.3f} ± {E['macro_f1'].std():.3f} | accuracy {E['acc'].mean():.3f} ± {E['acc'].std():.3f}")
        single = tab_all.loc[[s["label"] for s in nets_main], "macro_f1_mean"]
        print(f"Best single network macro-F1: {single.max():.3f} ({single.idxmax()}) -> ensemble gain {E['macro_f1'].mean() - single.max():+.3f}")

**Step 07 — Choose the app model.** Select by macro-F1 evidence, then use model size as the tie-breaker; save the matching final holdout run.

In [ ]:
# @title
APP_MODEL_OVERRIDE = None        # for example "Swin-T (pretrained)" to force a choice

if HAVE_CV:
    runs_all = pd.read_csv(RUNS_CSV)
    def run_params(label):
        s = next(x for x in stores.values() if x["label"] == label and x["scheme"] == MAIN_SCHEME)
        rn = s["folds"][min(s["folds"])]["run_name"]
        return float(runs_all.loc[runs_all["run_name"] == rn, "params_M"].iloc[0])

    candidates = [best] + tied
    choice = APP_MODEL_OVERRIDE or min(candidates, key=run_params)
    why = ("manual override" if APP_MODEL_OVERRIDE else
           "best mean macro-F1" if choice == best and not tied else
           f"smallest of the networks within one std of the best ({', '.join(candidates)})")
    sel = next(x for x in stores.values() if x["label"] == choice and x["scheme"] == MAIN_SCHEME)
    app_run = (f"{sel['arch']}{'-pre' if sel['pretrained'] else ''}_holdout_{sel['recipe']}{sel['tag']}_s{sel['seed']}")
    if app_run not in set(runs_all["run_name"]):
        raise RuntimeError("Train the selected architecture's final holdout run in Section 10 Step 01, then rerun this cell.")

    out = {"label": choice, "arch": sel["arch"], "pretrained": sel["pretrained"], "recipe": sel["recipe"],
           "tag": sel["tag"], "scheme": MAIN_SCHEME, "reason": why, "tied_with_best": tied,
           "macro_f1_mean": float(tab_all.loc[choice, "macro_f1_mean"]), "macro_f1_std": float(tab_all.loc[choice, "macro_f1_std"]),
           "acc_mean": float(tab_all.loc[choice, "acc_mean"]), "acc_std": float(tab_all.loc[choice, "acc_std"]),
           "app_run_name": app_run, "created": time.strftime("%Y-%m-%d %H:%M:%S")}
    (cfg.subdir("results") / "best_model.json").write_text(json.dumps(out, indent=2))
    tab_all.to_csv(cfg.subdir("results") / "final_comparison.csv")
    print("Model chosen for the app:", choice, "|", why)
    print(f"  macro-F1 {out['macro_f1_mean']:.3f} ± {out['macro_f1_std']:.3f} | accuracy {out['acc_mean']:.3f} ± {out['acc_std']:.3f}")
    print("  hold-out run for the app:", app_run or "NOT FOUND: train it with run_experiment(arch) first")
    print("Saved results/final_comparison.csv and results/best_model.json")

**Step 08 — Evaluate the selected model on the locked holdout test set (run once).** This cell reads the chosen run from `best_model.json`; it does not compare test scores across candidate models. Enable the flag only after model selection is complete.

In [ ]:
# @title
EVALUATE_SELECTED_HOLDOUT = True

if EVALUATE_SELECTED_HOLDOUT:
    best_path = cfg.subdir("results") / "best_model.json"
    if not best_path.exists():
        raise RuntimeError("Run Section 11 Step 07 first to select the app model.")
    selected_run = json.loads(best_path.read_text()).get("app_run_name")
    if not selected_run:
        raise RuntimeError("No final holdout run is linked to the selected model. Run final holdout training in Section 10.")

    model, card = load_run(MODELS_DIR / selected_run)
    pp = card["preprocessing"]
    prep = Preprocessor(target_size=pp["target_size"], crop_mode=pp["crop_mode"],
                        intensity_mode=pp["intensity_mode"], input_size=pp["input_size"])
    data = prepare_data("holdout", None, prep, verbose=False)
    assert abs(data["prep"].norm_mean - pp["norm_mean"]) < 1e-5, "normalisation differs from the model card"

    _, probs, labels, indices = model.evaluate(data["loaders"]["test"])
    np.savez(MODELS_DIR / selected_run / "holdout_test_preds.npz",
             probs=probs, labels=labels, index=indices)
    evaluator = Evaluator(cfg.class_names, model.threshold).set_data(probs, labels)
    holdout_summary = pd.DataFrame([{
        "run_name": selected_run,
        **evaluator.compute_all_metrics()["summary"],
        "flagged share": float(evaluator.flag_low_confidence().mean())
    }])
    holdout_summary.to_csv(cfg.subdir("results") / "holdout_test_results.csv", index=False)
    display(holdout_summary.round(3))
else:
    print("Locked holdout test not run. Enable once after Section 11 model selection.")

**Report questions.** Compare models with both baselines, explain per-class errors, assess split robustness, and state uncertainty honestly.

In [ ]:
# @title
# ===== published vs ours (4 classes) =====
PUBLISHED_4 = {"Paper: ResNet50": (53.0, 49.3, 49.8), "Paper: ResNeXt": (57.1, 54.0, 54.7),
               "Paper: DenseNet121": (58.2, 54.5, 55.5), "Paper: ViT (TinyViT)": (58.0, 55.7, 56.2),
               "Paper: EfficientNet": (61.9, 58.1, 59.6)}
rows = [{"model": k, "accuracy": v[0] / 100, "balanced acc": v[1] / 100, "macro-F1": v[2] / 100,
         "protocol": "10-fold, 70/30 split"} for k, v in PUBLISHED_4.items()]
ours = HEAD_MEAN.loc[MAIN_SCHEME].reset_index()
for _, r in ours.iterrows():
    rows.append({"model": "Ours: " + r["model"], "accuracy": r["acc"], "balanced acc": r["bal_acc"],
                 "macro-F1": r["macro_f1"], "protocol": "5-fold, vessel-grouped"})
PUB_CMP = pd.DataFrame(rows).sort_values("macro-F1", ascending=False).reset_index(drop=True)
display(PUB_CMP.round(3))
PUB_CMP.to_csv(cfg.subdir("results") / "published_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 0.42 * len(PUB_CMP) + 1.2))
colors = ["#9aa0a6" if m.startswith("Paper") else "#1a73e8" for m in PUB_CMP["model"]]
ax.barh(PUB_CMP["model"][::-1], PUB_CMP["macro-F1"][::-1], color=colors[::-1])
ax.set_xlabel("macro-F1 (four classes)")
ax.set_title("Published benchmarks (grey) vs this project (blue)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(cfg.subdir("figures") / "fig_published_comparison.png", dpi=130)
plt.show()

In [ ]:
# @title
# ===== three-class view (Fishing, Cargo, Tanker) =====
PUBLISHED_3 = {"Paper: ResNet50": (67.0, 64.4, 64.6), "Paper: ResNeXt": (68.3, 64.3, 64.9),
               "Paper: DenseNet121": (68.8, 66.4, 66.8), "Paper: ViT (TinyViT)": (66.8, 62.5, 63.8),
               "Paper: EfficientNet": (70.8, 67.8, 68.5)}
three = ["Fishing", "Cargo", "Tanker"]
idx3 = [cfg.class_names.index(c) for c in three]
y_all = df["label"].values
keep = np.isin(y_all, idx3)
pos = {c: i for i, c in enumerate(idx3)}
rows3 = [{"model": k, "accuracy": v[0] / 100, "balanced acc": v[1] / 100, "macro-F1": v[2] / 100}
         for k, v in PUBLISHED_3.items()]
for s in stores.values():
    if s["scheme"] != MAIN_SCHEME or not s["covered"].all():
        continue
    p3 = s["probs"][keep][:, idx3]
    y3 = np.array([pos[v] for v in y_all[keep]])
    m = Evaluator(three).set_data(p3, y3).compute_all_metrics()["summary"]
    rows3.append({"model": "Ours: " + s["label"], "accuracy": m["acc"], "balanced acc": m["bal_acc"], "macro-F1": m["macro_f1"]})
display(pd.DataFrame(rows3).sort_values("macro-F1", ascending=False).round(3).reset_index(drop=True))
print("Ours = the four-class models scored only on Fishing/Cargo/Tanker images, pooled out-of-fold. "
      "The paper trained separate three-class models, so this is an approximate comparison.")

In [ ]:
# @title
print(sig[sig["A"].str.startswith(("AIS", "Always")) | sig["B"].str.startswith(("AIS", "Always"))].round(3).to_string())

# Grad-CAM EXPLANABILITY
Generate CNN heatmaps and run sanity checks; Grad-CAM is an explanation aid, not proof of model reasoning.

**Step 01 — GradCAM class.** Capture feature maps and gradients from a spatial target layer and produce heatmap overlays.

In [ ]:
# @title
import torch.nn.functional as F

class GradCAM:
    """Class-diagram GradCAM: targetLayer, gradients, activations.

    Weights each feature map of the target layer by the average gradient of the class score,
    sums them, keeps the positive part and upsamples to the input size.
    """

    def __init__(self, net, target_layer, device, channels_last=False):
        self.net, self.target_layer, self.device = net, target_layer, device
        self.channels_last = channels_last
        self.gradients = self.activations = None
        self._handle = target_layer.register_forward_hook(self._on_forward)

    def _on_forward(self, module, inputs, output):
        self.activations = output
        if output.requires_grad:
            output.register_hook(lambda g: setattr(self, "gradients", g))

    def close(self):
        self._handle.remove()

    def compute_gradients(self, x, class_idx=None):
        """Forward pass, then backward pass of one class score. Returns (logits, class index)."""
        self.net.eval()
        self.gradients = self.activations = None
        x = x.to(self.device)
        with torch.enable_grad():
            self.net.zero_grad(set_to_none=True)
            logits = self.net(x)
            idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)
            logits[0, idx].backward()
        if self.gradients is None:
            raise RuntimeError("no gradients reached the target layer")
        return logits.detach(), idx

    def generate_heatmap(self, x, class_idx=None):
        """x: tensor (1, 1, H, W). Returns (heatmap (H, W) in [0, 1], class index, logits)."""
        logits, idx = self.compute_gradients(x, class_idx)
        A, G = self.activations.detach(), self.gradients.detach()
        if A.ndim != 4:
            raise RuntimeError(f"Grad-CAM needs a 4-D feature map, got shape {tuple(A.shape)}")
        if self.channels_last:
            A, G = A.permute(0, 3, 1, 2), G.permute(0, 3, 1, 2)
        weights = G.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * A).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tuple(x.shape[-2:]), mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), idx, logits

    @staticmethod
    def generate_heatmap_overlay(view, cam, alpha=0.45):
        """view: grayscale image in [0, 1] (H, W); cam: heatmap in [0, 1] (H, W). Returns an RGB uint8 image."""
        import matplotlib.pyplot as plt
        rgb = plt.get_cmap("jet")(cam)[..., :3]
        base = np.repeat(view[..., None], 3, axis=2)
        return (255 * ((1 - alpha) * base + alpha * rgb)).clip(0, 255).astype(np.uint8)

    @staticmethod
    def display_heatmap(view, cam, title=""):
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(view, cmap="gray")
        ax[0].set_title("model input")
        ax[1].imshow(GradCAM.generate_heatmap_overlay(view, cam))
        ax[1].set_title("Grad-CAM")
        for a in ax:
            a.axis("off")
        fig.suptitle(title)
        plt.show()


def get_target_layer(net, arch):
    """The last convolutional stage of each supported network."""
    if arch == "custom":
        return net.features[-1]
    if arch == "resnet18":
        return net.layer4[-1]
    if arch == "efficientnet_b0":
        return net.conv_head
    if arch == "swin_tiny":
        return net.layers[-1]            # channels-last (B, 7, 7, 768) in recent timm versions
    raise ValueError(f"No Grad-CAM target layer defined for architecture '{arch}'")


print('GradCAM ready.')

**Step 02 — Explain a saved model.** Rebuild preprocessing from the model card and inspect holdout examples and explanation checks.

In [ ]:
# @title
CNN_ARCHS = ("custom", "resnet18", "efficientnet_b0")
RUNS_CSV = cfg.subdir("results") / "runs.csv"
MODELS_DIR = cfg.subdir("models")

def pick_gradcam_run():
    runs = pd.read_csv(RUNS_CSV)
    bp = cfg.subdir("results") / "best_model.json"
    if bp.exists():
        b = json.loads(bp.read_text())
        if b.get("app_run_name") and b.get("arch") in CNN_ARCHS:
            return b["app_run_name"]
    final_suffix = f"_final_s{cfg.seed}"
    h = runs[(runs["scheme"] == "holdout") & runs["arch"].isin(CNN_ARCHS)
             & runs["run_name"].astype(str).str.endswith(final_suffix)]
    if h.empty:
        raise RuntimeError("No CNN hold-out run found. Train one with run_experiment('custom') in Section 08.")
    return h.sort_values("val_macro_f1", ascending=False)["run_name"].iloc[0]

def explain_run(run_name, n_check=60, tag="cnn", seed=42):
    model, card = load_run(MODELS_DIR / run_name)
    pp, arch = card["preprocessing"], card["run_config"]["arch"]
    prep_e = Preprocessor(target_size=pp["target_size"], crop_mode=pp["crop_mode"],
                          intensity_mode=pp["intensity_mode"], input_size=pp["input_size"])
    d = prepare_data("holdout", None, prep_e, verbose=False)
    loader = d["loaders"]["test"]
    ds = loader.dataset
    _, probs, y, idx = model.evaluate(loader)
    pred, conf = probs.argmax(1), probs.max(1)
    mean, std = prep_e.norm_mean, prep_e.norm_std
    gc = GradCAM(model.net, get_target_layer(model.net, arch), device, channels_last=(arch == "swin_tiny"))
    rng = np.random.default_rng(seed)

    def view_of(x):
        v = x[0].numpy() * std + mean
        lo, hi = np.percentile(v, (1.0, 99.5))
        return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1)

    try:
        # ---------- gallery ----------
        picks = []
        for c in range(cfg.num_classes):
            ok, bad = np.where((y == c) & (pred == c))[0], np.where((y == c) & (pred != c))[0]
            picks += list(ok[np.argsort(-conf[ok])][:2]) + list(bad[np.argsort(-conf[bad])][:1])
        per_row = 4
        rows = int(np.ceil(len(picks) / per_row))
        fig, axes = plt.subplots(2 * rows, per_row, figsize=(3.2 * per_row, 6.4 * rows), squeeze=False)
        for a in axes.ravel():
            a.axis("off")
        for n, j in enumerate(picks):
            r, c_ = divmod(n, per_row)
            x = ds[j][0]
            cam, _, _ = gc.generate_heatmap(x[None], int(pred[j]))
            v = view_of(x)
            axes[2 * r, c_].imshow(v, cmap="gray")
            axes[2 * r, c_].set_title(f"{cfg.class_names[y[j]]} -> {cfg.class_names[pred[j]]} ({conf[j]:.2f})", fontsize=8)
            axes[2 * r + 1, c_].imshow(GradCAM.generate_heatmap_overlay(v, cam))
        fig.suptitle(f"Grad-CAM on hold-out test images: {run_name}", fontsize=10)
        plt.tight_layout()
        fname = "fig_gradcam_examples.png" if tag == "cnn" else f"fig_gradcam_examples_{tag}.png"
        plt.savefig(cfg.subdir("figures") / fname, dpi=130)
        plt.show()

        # ---------- checks ----------
        sel = rng.choice(len(ds), min(n_check, len(ds)), replace=False)
        H = ds[0][0].shape[-1]
        k = int(0.2 * H * H)
        lo_c = (H - 96) // 2
        drops_cam, drops_rnd, corr, centre = [], [], [], []
        for j in sel:
            x = ds[j][0]
            c1, c2 = int(pred[j]), int(np.argsort(-probs[j])[1])
            cam1, _, _ = gc.generate_heatmap(x[None], c1)
            cam2, _, _ = gc.generate_heatmap(x[None], c2)
            corr.append(float(np.corrcoef(cam1.ravel(), cam2.ravel())[0, 1]))
            centre.append(float(cam1[lo_c:lo_c + 96, lo_c:lo_c + 96].sum() / (cam1.sum() + 1e-8)))
            base_p = float(model.predict_proba(x[None])[0, c1])
            m_cam = np.zeros(H * H, dtype=bool)
            m_cam[np.argpartition(-cam1.ravel(), k)[:k]] = True
            m_rnd = np.zeros(H * H, dtype=bool)
            m_rnd[rng.choice(H * H, k, replace=False)] = True
            for m, store in ((m_cam, drops_cam), (m_rnd, drops_rnd)):
                x2 = x[None].clone()
                x2[0, 0][torch.from_numpy(m.reshape(H, H))] = 0.0       # 0 = training-set mean level
                store.append(base_p - float(model.predict_proba(x2)[0, c1]))
    finally:
        gc.close()

    stats = {"run": run_name, "arch": arch, "n_checked": int(len(sel)),
             "deletion_drop_gradcam": float(np.mean(drops_cam)), "deletion_drop_random": float(np.mean(drops_rnd)),
             "share_gradcam_beats_random": float(np.mean(np.array(drops_cam) > np.array(drops_rnd))),
             "corr_predicted_vs_runnerup": float(np.nanmean(corr)),
             "centre_mass_fraction": float(np.mean(centre)), "centre_area_fraction": float((96 / H) ** 2)}
    p = cfg.subdir("results") / "gradcam_checks.json"
    allstats = json.loads(p.read_text()) if p.exists() else {}
    allstats[tag] = stats
    p.write_text(json.dumps(allstats, indent=2))
    return stats

def print_gradcam_stats(s):
    print(f"Run {s['run']} ({s['arch']}), {s['n_checked']} hold-out test images")
    print(f"1. Deletion test: hiding the top 20% of the heatmap lowers the predicted probability by "
          f"{s['deletion_drop_gradcam']:.3f}, hiding 20% random pixels by {s['deletion_drop_random']:.3f}; "
          f"the heatmap is the more damaging in {s['share_gradcam_beats_random'] * 100:.0f}% of images.")
    print(f"2. Class specificity: correlation between the predicted-class map and the runner-up map "
          f"{s['corr_predicted_vs_runnerup']:.2f} (near 1 means the map ignores the class).")
    print(f"3. Ship-centred attention: {s['centre_mass_fraction'] * 100:.0f}% of the heatmap lies in the central window, "
          f"which covers {s['centre_area_fraction'] * 100:.0f}% of the image.")
print("Explain helpers ready.")

**Step 03 — Explain the selected CNN.** Set the CNN run name or use the automatic CNN selection, then review the checks.

In [ ]:
# Explain the selected CNN
GRADCAM_RUN = pick_gradcam_run()
cnn_stats = explain_run(GRADCAM_RUN, tag="cnn")
print("CNN run:", GRADCAM_RUN)
print_gradcam_stats(cnn_stats)

**Step 04 — Optional Swin heatmap.** Try the final Swin stage only when its feature layout is supported by the installed `timm` version.

In [ ]:
# Optional Swin heatmap
GRADCAM_ALSO_SWIN = True
if GRADCAM_ALSO_SWIN:
    swin_run = f"swin_tiny-pre_holdout_log_resize_256_final_s{cfg.seed}"
    swin_stats = explain_run(swin_run, tag="swin")
    print_gradcam_stats(swin_stats)
else:
    print("Swin Grad-CAM skipped.")